# ⚡ 1교시: Spark를 넘어서, 왜 Flink인가?

## 🎯 학습 목표
- Spark Structured Streaming의 한계 이해
- Micro-batch vs Native Streaming 차이 파악
- Apache Flink의 핵심 특징 학습
- PyFlink의 두 가지 API 개요 파악

---
## 1. 우리가 구축했던 파이프라인 복습

2차 프로젝트에서 **Kafka(수집) → Spark(배치/실시간 처리) → Elasticsearch(저장)** 파이프라인을 경험

**질문**: 만약 1초의 지연도 허용하지 않는 시스템을 만든다면?
- 금융 사기 탐지 (Fraud Detection)
- 실시간 게임 아이템 추천
- 자율주행 센서 데이터 처리

→ 지금의 Spark 방식이 최선일까?

=== 우리가 만든 파이프라인 ===

```
[Kafka Producer] → [Kafka Broker] → [Spark Structured Streaming] → [Elasticsearch]
    (수집)            (저장/전달)         (실시간? 처리)                (검색/시각화)
```

✅ 잘 동작함
✅ 대용량 처리 가능

❓ 하지만...
- "실시간"이라고 했는데, 정말 실시간인가?
- Spark는 데이터를 "모아서" 처리하는 구조
- 1초 미만의 반응이 필요한 상황에서는?

---
## 2. 결정적 차이: Micro-batch vs Native Streaming

### Spark Structured Streaming (기존에 배운 것)

- **방식**: **마이크로 배치**(Micro-batching)
- 데이터를 실시간으로 처리하는 것처럼 보이지만, 사실은 **매우 짧은 주기로 데이터를 모아서 한 번에 처리**
- 비유: **버스** → 승객을 모아서 출발
- 아무리 빨라도 데이터를 모으는 시간(Latency)이 존재
- "거의" 실시간이지 "완전한" 실시간은 아님

### Apache Flink (오늘 배울 것)

- **방식**: **이벤트 기반**(Event-driven) / **네이티브 스트리밍**(Native Streaming)
- 버스를 기다리지 않고, 승객이 오자마자 바로 출발하는 **전용 택시** 또는 **수도관(Pipe)**
- 데이터가 들어오는 즉시(Row by Row) 처리
- 밀리세컨드(ms) 단위의 초저지연 처리 가능

=== Micro-batch vs Native Streaming ===
![Micro-batch vs Native Streaming](https://cdn.discordapp.com/attachments/1457516082071081045/1470622787440869376/ChatGPT_Image_2026_2_10_12_28_47.png?ex=698bf7b3&is=698aa633&hm=00e6d2017780871f3b42b6ad25d8f5f3a86510032254a335f40a427ebed6e2b0&)

---
## 2.1 배치 / 실시간 / 스트림 개념 정리

**❓ Kafka와 Flink는 실시간 처리 도구인가요?**
- 엄밀히 말하면 **스트림 처리(Stream Processing)** 도구입니다.
- 스트림 처리를 통해 실시간(Real-time) 성능을 "구현"하는 것입니다.

### 📊 비교표
| 구분 | 배치(Batch) | 실시간(Real-time) | 스트림(Stream) |
|:---|:---|:---|:---|
| **한줄 요약** | 모아서 한 번에 처리 | 즉시 처리 | 흘러오는 대로 계속 처리 |
| **비유** | 세탁물 모아서 빨래 | 손빨래 즉시 | 컨베이어 벨트 자동 세탁 |
| **지연 시간** | 분 ~ 시간 | 밀리초 ~ 초 | 초 ~ 분 |
| **대표 도구** | Spark Batch, Airflow | OLTP DB | Kafka, Flink |

### 🧠 쉽게 기억하기
- **배치**: "나중에 한꺼번에" → 급하지 않은 대량 작업 (유한 데이터)
- **실시간**: "지금 당장" → 즉각 응답이 필요한 비즈니스 요구사항
- **스트림**: "끊임없이 흘러가며" → 데이터가 계속 들어오는 처리 방식 (무한 데이터)

> **💡 핵심 요약**: 실시간은 **속도(SLA)** 에 대한 요구사항이고, 스트림은 **데이터 처리 패턴**입니다. Flink는 스트림 처리 방식을 사용하여 실시간 요구사항을 완벽하게 충족합니다.

---
## 3. Spark에서 까다로웠던 문제들, Flink는 이렇게 풉니다

### ① 늦게 도착한 데이터 처리 (Event Time & Watermark)

**상황**: 네트워크 지연으로 10:00의 데이터가 10:05에 도착

| 구분 | Spark | Flink |
|------|-------|-------|
| 워터마크 갱신 시점 | **마이크로 배치 경계**에서만 갱신 | 데이터 흐름 속 **특수 이벤트**로 실시간 갱신 |
| 윈도우 닫기 | 배치가 완료되어야 확정 | 워터마크 도달 즉시 닫기 |
| 늦은 데이터 처리 | 워터마크 이후 데이터 → 버림 | **Side Output**으로 별도 추출 → 재처리 가능 |

### ② 상태 관리 (State Management)

**💡 상태(State)란?**

스트림 처리 중 **과거의 데이터를 기억**해야 할 때 저장하는 정보입니다.

(예: 현재까지의 합계, 최근 1시간 동안의 접속 기록, 머신러닝 모델 파라미터 등)

**상황**: "지난 7일 동안 이 유저가 접속한 횟수 누적" 같은 긴 호흡의 데이터

| 구분 | Spark | Flink |
|------|-------|-------|
| 저장 위치 | **메모리(Heap)** 기본 | **메모리** 또는 **RocksDB(Disk)** 선택 가능 |
| 대용량 상태 | JVM 메모리 압박 심함 | RocksDB로 PB급 상태 관리 가능 |
| 체크포인트 | 전체 상태 디스크 기록 (Stop-the-world) | **증분 백업**(Incremental Checkpointing) 지원 |

---
## 4. 비교 정리표

| 비교 포인트 | Spark Structured Streaming (배운 것) | Apache Flink (배울 것) | Flink가 필요한 순간 |
|-------------|-------------------------------------|----------------------|-------------------|
| **처리 방식** | **Micro-batch** (모아서 처리) | **Native Streaming** (오자마자 처리) | 1초 미만의 즉각적인 반응이 생명일 때 |
| **지연 시간** | 수 초(Seconds) 단위 | 밀리초(Milliseconds) 단위 | 금융, 이상 탐지, 실시간 관제 |
| **데이터 처리 단위** | RDD / DataFrame (작은 배치) | Event (개별 데이터) | 데이터 하나하나의 순서가 중요할 때 |
| **상태 관리** | 배치 주기에 의존적 | 매우 정교하고 강력함 (Stateful) | 복잡한 시계열 패턴 분석 |


**언제 Spark, 언제 Flink?**

✅ Spark가 적합한 경우:
   - 대용량 배치 처리 (ETL)
   - 수 초 지연 허용 가능한 실시간 처리
   - 풍부한 생태계(ML, Graph 등) 활용
   - 팀에 Spark 경험자가 많을 때

✅ Flink가 적합한 경우:
   - 밀리초 단위 응답 필수 (금융 사기 탐지)
   - 복잡한 이벤트 패턴 감지 (CEP)
   - 대규모 상태 관리 (유저별 세션 추적)
   - 정확한 이벤트 시간 처리 필요
   - 실시간 관제/모니터링 시스템

💡 결론:
   "Spark는 대용량 데이터를 빠르게 처리하는 데 여전히 강력한 도구.
    하지만 '반응 속도'가 비즈니스의 핵심 가치인 영역에서는 Flink가 표준."

---
## 5. Apache Flink란?

![Flink Logo](https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQu3LiaVLcRpXcG3ysBB9S41C_bhZT_lXFshQ&s)

### 정의
- **Apache Flink**: 분산 스트림 처리 엔진
- 실시간 데이터 스트리밍 + 배치 처리 모두 지원
- **"모든 것은 스트림"** → 배치는 유한한 스트림(Bounded Stream)의 특수 케이스

### 핵심 특징

| 특징 | 설명 |
|------|------|
| **True Streaming** | 마이크로배치가 아닌 진정한 스트림 처리 |
| **Event Time** | 이벤트 발생 시간 기반 처리 |
| **Stateful** | 상태 관리 및 Exactly-Once 보장 |
| **Fault Tolerant** | 체크포인트 기반 장애 복구 |
| **Unified API** | 스트림/배치 통합 API |

> 📖 **공식 문서**: [Apache Flink](https://flink.apache.org/) 참고

---
## 6. PyFlink 개요

**PyFlink**: Apache Flink의 Python API

![pyflink](https://nightlies.apache.org/flink/flink-docs-release-2.2/fig/pyflink.svg)

### 주요 활용 분야
- 실시간 데이터 처리 파이프라인
- 대규모 탐색적 데이터 분석
- 머신러닝(ML) 파이프라인
- ETL 프로세스

### 장점
- Python과 Pandas에 익숙하다면 Flink 생태계를 쉽게 활용 가능

### 두 가지 API

| API | 특징 | Spark 대응 |
|-----|------|-----------|
| **Table API** | SQL과 유사, 높은 수준의 추상화, 자동 최적화 | **DataFrame API** |
| **DataStream API** | 상태(state)와 시간(time)에 대한 저수준 제어, 세밀한 제어 | **RDD API** |

→ 선언적이고 SQL 스타일 작업 → **Table API**

→ 복잡한 스트리밍 로직과 세부 제어 → **DataStream API**

> 📖 **공식 문서**: [PyFlink Documentation](https://nightlies.apache.org/flink/flink-docs-stable/docs/dev/python/overview/) 참고


---
## 📝 FAQ

**Q1. Flink가 Spark보다 항상 좋은 건가요?**

→ 아닙니다. Spark는 대용량 배치 처리, ML 파이프라인, 풍부한 생태계에서 여전히 강력합니다. **"반응 속도"가 핵심인 영역**에서만 Flink가 유리합니다.

**Q2. Flink도 배치 처리를 할 수 있나요?**

→ 네. Flink는 "모든 것은 스트림"이라는 철학으로, 배치를 "유한한 스트림"으로 처리합니다. `RuntimeExecutionMode.BATCH` 설정으로 전환 가능합니다.

**Q3. PyFlink는 PySpark보다 느린가요?**

→ PyFlink의 Table API는 Java 기반 최적화 엔진을 사용하므로 성능 차이가 크지 않습니다. DataStream API에서 Python UDF를 사용할 때는 JVM-Python 간 통신 오버헤드가 발생할 수 있습니다.

**Q4. Flink의 학습 곡선이 높다고 하던데요?**

→ Spark를 이미 배웠기 때문에 많은 개념이 겹칩니다. Source, Transformation, Sink 구조는 동일하고, 용어만 다릅니다.

**Q5. 실무에서 Flink를 많이 사용하나요?**

→ 넷플릭스, 우버, 알리바바 등 대규모 실시간 처리가 필요한 기업에서 광범위하게 사용 중입니다. 한국에서도 카카오, 네이버, 쿠팡 등에서 활용합니다.

---
## ✅ 퀴즈

**Q1.** Spark Structured Streaming의 처리 방식을 가장 정확하게 설명한 것은?

- A) 이벤트가 도착하는 즉시 하나씩 처리하는 방식
- B) 데이터를 짧은 주기로 모아서 배치로 처리하는 방식
- C) 데이터를 메모리에 전부 올린 후 한 번에 처리하는 방식
- D) GPU를 활용하여 병렬 처리하는 방식

<details>
<summary>정답 확인</summary>

**정답: B)**

Spark Structured Streaming은 **마이크로 배치(Micro-batch)** 방식으로, 데이터를 짧은 주기(예: 100ms~수 초)로 모아서 한 번에 처리합니다. 실시간처럼 보이지만 완전한 실시간은 아닙니다.

</details>

---

**Q2.** Flink가 Spark보다 유리한 상황으로 **적절하지 않은** 것은?

- A) 금융 사기 탐지 시스템
- B) 대용량 로그 데이터의 일괄 ETL 처리
- C) 실시간 게임 아이템 추천
- D) IoT 센서 이상 탐지

<details>
<summary>정답 확인</summary>

**정답: B)**

대용량 로그 데이터의 일괄 ETL 처리는 **배치 처리**에 해당하며, 이는 Spark의 강점입니다. Flink는 밀리초 단위의 즉각적 반응이 필요한 A, C, D 같은 상황에서 유리합니다.

</details>

---

**Q3.** PyFlink의 Table API와 DataStream API에 대한 설명으로 올바른 것은?

- A) Table API는 저수준 제어에 적합하다
- B) DataStream API는 SQL과 유사한 방식이다
- C) Table API는 Spark의 DataFrame API와 유사하다
- D) DataStream API만 배치 처리를 지원한다

<details>
<summary>정답 확인</summary>

**정답: C)**

Table API는 SQL과 유사한 **고수준 추상화**로, Spark의 **DataFrame API**와 대응됩니다. DataStream API가 저수준 제어(Spark RDD 대응)에 해당합니다.

</details>

---
## 📌 핵심 요약

| 개념 | 핵심 내용 |
|------|----------|
| **Micro-batch** | Spark 방식. 데이터를 모아서 처리. 수 초 지연 |
| **Native Streaming** | Flink 방식. 이벤트 즉시 처리. 밀리초 지연 |
| **Watermark** | Flink는 데이터 흐름 속 특수 이벤트로 처리, Side Output으로 늦은 데이터 재처리 가능 |
| **State Management** | Flink는 RocksDB로 PB급 상태 관리, 증분 백업 지원 |
| **PyFlink Table API** | SQL 스타일, 고수준 추상화, Spark DataFrame 대응 |
| **PyFlink DataStream API** | 저수준 제어, 복잡한 이벤트 처리, Spark RDD 대응 |

# ⚡ PyFlink 입문: 환경 설정부터 DataStream & Table API까지

이 교안은 PyFlink의 핵심 개념과 API를 실습합니다.
Day27 1교시에서 Spark와 Flink의 차이, PyFlink 개요를 살펴보았습니다.
이제 직접 코드를 작성하면서 PyFlink의 두 가지 API를 체험해 봅시다.

## 🎯 학습 목표

1. PyFlink 실행 환경(Docker)을 이해하고 설정할 수 있다
2. 고차함수(Higher-Order Function) 개념을 이해하고 DataStream API에 적용할 수 있다
3. DataStream API로 데이터 변환 파이프라인을 구성할 수 있다
4. State(상태)와 Window(윈도우)를 활용한 스트림 처리를 구현할 수 있다
5. Table API로 SQL 스타일의 데이터 처리를 수행할 수 있다
6. DataStream API와 Table API를 혼합하여 사용할 수 있다

**전제조건**
- **Day27 1교시** 내용 복습 (Spark vs Flink 비교, PyFlink 개요)
- Docker 기본 사용법 (이미지 빌드, 컨테이너 실행)
- Python 기초 문법 (함수 정의, 리스트 조작)

**실행 환경**: Docker 컨테이너 (`my-pyflink-image`) 내부 Jupyter에서 실행

> 📖 **참고 문서**: [Flink Python API 공식 문서](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/overview/)

---
## 1. PyFlink 환경 설정

Docker에 대한 기본 수업은 이미 진행했습니다. 여기서는 **"왜 Docker?"**가 아니라
**"어떻게 Docker를 활용해서 Flink를 실행할 것인가?"**에 초점을 맞춥니다.

PyFlink를 설치하려면 단순한 `pip install`이 아니라, Java(JVM)와 Python 사이의
**C 브릿지 컴파일**이 필요합니다. 이 복잡한 과정을 Docker로 깔끔하게 해결합니다.

### 1.1 단일 Dockerfile 해설

"도대체 왜 이렇게 복잡해?" — PyFlink 빌드의 3가지 핵심 포인트를 풀어봅니다.

**① 왜 `FROM python`이 아니라 `FROM flink`인가?**

일반적인 Python 앱에서는 Python이 주인이고, 필요한 라이브러리를 가져옵니다.
하지만 PyFlink에서는 **Java(JVM)가 주인**이고, Python은 "얹혀사는" 구조입니다.

Flink는 거대한 분산 처리 시스템으로, 클러스터 관리나 메모리 처리가 전부 Java/Scala로
작성되어 있습니다. 그래서 "Python 이미지에 Java를 설치하는 것"보다
**"이미 완벽하게 세팅된 Flink(Java) 이미지에 Python을 끼워 넣는 것"**이 훨씬 안정적입니다.

**② `build-essential`(gcc)과 `python3-dev`는 왜 필요한가?**

단순한 Python 라이브러리가 아니기 때문입니다.
PyFlink가 실행될 때, Python 코드는 Java(JVM)와 실시간으로 데이터를 주고받아야 합니다.
이때 **Pemja** 같은 기술을 쓰는데, 이건 Python과 Java 사이를 이어주는 **다리(Bridge)** 역할을 합니다.
이 다리는 **C/C++**로 만들어져 있습니다. 즉, `pip install apache-flink`를 하는 순간,
내부적으로는 **"C 코드를 컴파일해서 다리를 짓는 공사"**가 일어납니다.
공사를 하려니 **공구(gcc 컴파일러)**와 **재료(python-dev 헤더)**가 반드시 필요한 것입니다.

**③ 가장 이해 안 가는 부분: `default-jdk`와 `JAVA_HOME` 변경**

"Java 깔려 있다며? 근데 왜 또 JDK를 깔고 경로를 바꿔?"

- **Flink 기본 이미지의 상태 (JRE)**: "실행(Run)"에 최적화. 자동차로 치면 **운전석만 있는 상태(JRE)**
- **PyFlink 설치의 요구사항**: "다리 공사(컴파일)"를 하려면 Java의 내부 부품 규격서(`jni.h`)가 필요.
  이 규격서는 **공장(JDK)**에만 있음
- **해결책**: `apt-get install default-jdk`로 임시 공장 설치 →
  `JAVA_HOME=/usr/lib/jvm/default-java`로 설치할 때만 공장(JDK 경로)을 보도록 지정

만약 이 설정을 안 하면, 설치 프로그램은 운전석(JRE)에서 부품 규격서를 찾다가
"파일이 없는데요?" 하고 에러를 뱉게 됩니다.

### 단일 Dockerfile 코드

```dockerfile
# 1. Base Image: Flink 공식 이미지 사용 (Java 기반)
FROM flink:latest

# 2. 패키지 설치를 위한 관리자 권한 부여
USER root

# 3. 필수 의존성 설치
# - python3, python3-dev: Python 환경 및 개발 헤더
# - build-essential: PyFlink-Java 간 통신(pemja) 컴파일을 위한 gcc 포함
# - default-jdk: JNI 헤더(jni.h) 포함 (빌드 시 필수)
RUN apt-get update && apt-get install -y \
    python3 \
    python3-dev \
    build-essential \
    default-jdk \
    && ln -s /usr/bin/python3 /usr/bin/python \
    && rm -rf /var/lib/apt/lists/*

# 4. 패키지 매니저 uv 설치
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

# 5. PyFlink 설치 (핵심 빌드 과정)
# 설치 중에만 JAVA_HOME을 JDK 경로로 임시 지정
RUN JAVA_HOME=/usr/lib/jvm/default-java \
    uv pip install --system --break-system-packages apache-flink

# 6. 기본 사용자(flink)로 권한 복귀
USER flink
WORKDIR /opt/flink
```

### 1.2 멀티 스테이지 빌드 (Multi-stage Build)

![](https://labs.iximiuz.com/content/files/tutorials/docker-multi-stage-builds/__static__/multi-stage-build.png   )

이전 설명에서 "다리 공사(컴파일)"를 위해 `gcc`와 `JDK`라는 **중장비**가 필요하다고 했습니다.
멀티 스테이지 빌드를 쓰면, **"공사장에서는 중장비를 쓰고, 완공된 다리(이미지)만
헬기로 떠서 깨끗한 새 땅(Final Stage)에 옮겨 놓는 것"**이 가능해집니다.

**Stage 1 (Builder) — 공사 현장**

- JDK 깔고, GCC 깔고, Python 헤더 파일들 설치
- `uv pip install`로 C 코드를 컴파일해서 다리를 건설
- 결과물은 `/opt/venv`라는 상자 안에 담김

**COPY (이사)**

- `COPY --from=builder /opt/venv /opt/venv`
- **"공사 현장(Builder)에서 연장 챙기지 말고, 완성된 물건(venv)만 새 집으로 가져와!"**
- 무거운 JDK, GCC, 소스코드 찌꺼기는 Stage 1에 버려두고 옴

**Stage 2 (Runner) — 입주 완료**

- 새 집에는 **Python 실행기(interpreter)**만 설치
- 부품을 만드는 기계(JDK/GCC)는 더 이상 필요 없음. 이미 부품(PyFlink)은 완성됨

**결과 비교**

| 구분 | 단일 빌드 (이전 방식) | 멀티 스테이지 빌드 (최적화) |
|------|----------------------|--------------------------|
| **포함 요소** | Python + PyFlink + **JDK** + **GCC** + **Headers** | Python + PyFlink |
| **이미지 크기** | 매우 큼 (JDK/GCC만으로 수백 MB 추가) | **가벼움** (필요한 것만 있음) |
| **빌드 속도** | 빠름 (한 번에 진행) | 비슷하거나 약간 느림 |
| **보안** | 컴파일러(GCC)가 있어서 해킹 시 위험 요소 | 컴파일러가 아예 없어서 더 안전 |

**결론:** 프로덕션(실무) 환경에 배포할 목적이라면, 이 **멀티 스테이지 방식**이 정석입니다.

### 멀티 스테이지 Dockerfile 코드

```dockerfile
# ==========================================
# [Stage 1] Builder: 공장 (빌드 도구가 다 있음)
# ==========================================
FROM flink:latest AS builder

USER root

# 1. 빌드용 중장비 설치 (JDK, GCC, Python Dev)
RUN apt-get update && apt-get install -y \
    python3 \
    python3-dev \
    python3-venv \
    build-essential \
    default-jdk \
    && rm -rf /var/lib/apt/lists/*

# 2. uv 설치
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

# 3. 가상환경 생성 및 PyFlink 설치
ENV VIRTUAL_ENV=/opt/venv
RUN python3 -m venv $VIRTUAL_ENV
ENV PATH="$VIRTUAL_ENV/bin:$PATH"

# 4. PyFlink 빌드 (JDK 헤더 필요)
RUN JAVA_HOME=/usr/lib/jvm/default-java \
    uv pip install apache-flink

# ==========================================
# [Stage 2] Runner: 쇼룸 (완성품만 있음)
# ==========================================
FROM flink:latest

USER root

# 1. 런타임 필수 요소만 설치
RUN apt-get update && apt-get install -y \
    python3 \
    && ln -s /usr/bin/python3 /usr/bin/python \
    && rm -rf /var/lib/apt/lists/*

# 2. Builder에서 조립이 끝난 라이브러리 폴더만 복사
COPY --from=builder /opt/venv /opt/venv

# 3. 환경 변수 설정
ENV VIRTUAL_ENV=/opt/venv
ENV PATH="$VIRTUAL_ENV/bin:$PATH"

USER flink
WORKDIR /opt/flink
```

<!-- IMAGE: Docker build 과정 또는 이미지 크기 비교 스크린샷 -->

In [7]:
# PyFlink 설치 확인
import pyflink
print(f"PyFlink 버전: {pyflink.__version__}")

PyFlink 버전: 2.2.0


In [8]:
# 공통 import
import json
from typing import Iterable

# DataStream API imports
from pyflink.common import Types, WatermarkStrategy, Time, Encoder, Row, Duration
from pyflink.common.time import Instant
from pyflink.common.watermark_strategy import TimestampAssigner
from pyflink.datastream import StreamExecutionEnvironment, RuntimeExecutionMode
from pyflink.datastream.functions import KeyedProcessFunction, RuntimeContext
from pyflink.datastream.state import ValueStateDescriptor, StateTtlConfig

# Table API imports
from pyflink.table import (
    TableEnvironment, StreamTableEnvironment, EnvironmentSettings,
    TableDescriptor, Schema, DataTypes, FormatDescriptor
)
from pyflink.table.expressions import col, lit, row_interval, CURRENT_ROW
from pyflink.table.udf import udf, udtf, udaf
from pyflink.table.window import Tumble, Slide, Session, Over

print("모든 모듈이 성공적으로 import 되었습니다!")

모든 모듈이 성공적으로 import 되었습니다!


### 1.3 Flink UI 모니터링

Flink 작업을 실행하면 로컬 대시보드(기본값: http://localhost:8081)를 통해
작업의 진행 상황을 모니터링할 수 있습니다.

- **Jobs**: 현재 실행 중인 Job과 완료된 Job의 상태(Running, Finished, Failed 등) 확인
- **Visualized Graph**: 작성한 코드가 어떻게 실행 파이프라인으로 구성되었는지 시각화
- **Task Metrics**: 각 연산자별 처리 데이터 개수, 처리량, 지연 시간 확인
- **Exceptions**: 작업 실패 시 원인이 되는 Stacktrace 상세 확인
- **Watermarks**: 이벤트 시간 기반 처리 시 Watermark 갱신 상태 확인
- **Checkpoints**: 상태 관리 사용 시 체크포인트 수행 여부 및 상태 크기 모니터링

<!-- IMAGE: Flink Web UI 대시보드 스크린샷 -->

### 📝 FAQ

**Q. `my-pyflink-image`는 어떻게 빌드하나요?**

> 위 Dockerfile을 `Dockerfile`이라는 이름으로 저장한 뒤, 터미널에서
> `docker build -t my-pyflink-image .` 명령을 실행하면 됩니다.
> 멀티 스테이지 버전은 `Dockerfile.Multi`로 저장해서 별도 빌드할 수 있습니다.

**Q. 멀티스테이지 빌드를 꼭 써야 하나요?**

> 학습 단계에서는 단일 빌드로 충분합니다. 프로덕션 배포 시에는 이미지 크기와 보안을
> 위해 멀티 스테이지 빌드를 권장합니다. 단일 빌드 대비 수백 MB의 용량을 절약할 수 있습니다.

**Q. Flink UI는 어디서 보나요?**

> Docker로 Flink 클러스터를 실행하면 기본적으로 `http://localhost:8081`에서
> Web UI에 접근할 수 있습니다. `compose.yml`에서 포트 매핑을 확인하세요.

---

### 🏋️ 실습

제공된 Dockerfile로 이미지를 빌드한 후, `docker images` 명령으로
단일 빌드와 멀티 스테이지 빌드의 이미지 크기 차이를 확인해 보세요.

```bash
# 단일 빌드
docker build -t pyflink-single -f Dockerfile .

# 멀티 스테이지 빌드
docker build -t pyflink-multi -f Dockerfile.Multi .

# 크기 비교
docker images | grep pyflink
```

---
## 2. 고차함수 — DataStream API를 위한 준비

### 왜 갑자기 "고차함수"를 알아야 할까?

Spark를 사용하셨다면 아마 이런 코드가 가장 익숙하실 겁니다.

```python
# Spark DataFrame API — 우리가 익숙한 방식
df.select("name", "price") \
  .filter(col("price") > 20000) \
  .groupBy("category") \
  .agg(sum("price"))
```

DataFrame API는 **"무엇을 원하는지"만 선언**하면 Spark 엔진이 알아서
실행 계획을 짜주는 방식입니다. 내부적으로 어떤 순서로 돌아가는지,
데이터를 한 건씩 어떻게 처리하는지 신경 쓸 필요가 없었습니다.

그런데 Flink의 DataStream API로 오면 분위기가 달라집니다.

```python
# Flink DataStream API — 앞으로 배울 방식
ds.flat_map(split) \
  .map(lambda i: (i, 1)) \
  .key_by(lambda i: i[0]) \
  .reduce(lambda i, j: (i[0], i[1] + j[1]))
```

언뜻 비슷해 보이지만, 결정적인 차이가 하나 있습니다.
`map()` 안에 **컬럼 이름이 아니라 "함수"가 들어가 있다**는 것입니다.
이 함수를 매개변수로 넘기는 방식을 이해하는 것이 DataStream API의 출발점이고,
그 개념이 바로 **"고차 함수(Higher-Order Function)"**입니다.

### DataFrame API와 무엇이 다른가?

차이를 가장 쉽게 보여주는 예시가 있습니다.
**"가격에 10% 세금을 붙여라"**라는 같은 작업을 두 방식으로 해보겠습니다.

```python
# Spark DataFrame API 방식
df.withColumn("price_with_tax", col("price") * 1.1)
```

이 코드에서 우리는 **"어떤 컬럼에 어떤 연산을 할지"를 표현식(Expression)으로 전달**했습니다.
`col("price") * 1.1`은 함수가 아니라, Spark가 해석할 수 있는 "계산 명세서" 같은 것입니다.

```python
# Flink DataStream API 방식
ds.map(lambda record: record * 1.1)
```

이 코드에서는 **"데이터 한 건이 들어오면 어떻게 처리할지"를 함수로 직접 전달**했습니다.
`lambda record: record * 1.1`은 진짜 Python 함수입니다.

| | Spark DataFrame API | Flink DataStream API |
|---|---|---|
| 전달하는 것 | 컬럼 표현식 (`col("price") * 1.1`) | **함수** (`lambda x: x * 1.1`) |
| 엔진의 역할 | 표현식을 해석하고 최적화까지 담당 | 전달받은 함수를 그대로 실행 |
| 비유 | "이 서류대로 처리해주세요" (위임) | "이 도구로 직접 깎으세요" (직접 수행) |

이 차이 때문에, DataStream API를 쓰려면 **"함수를 다른 함수에 넘긴다"**는 개념을 먼저 이해해야 합니다.

### 먼저 기억할 것: Python에서 함수는 "값"이다

고차함수를 이해하려면, 먼저 이 사실을 받아들여야 합니다.

> **Python에서 함수는 숫자나 문자열처럼 변수에 담을 수 있는 "값"이다.**

DataFrame API에서는 이런 걸 생각할 필요가 없었습니다.
`col("price")`라는 표현식을 넘기면 끝이었으니까요.
하지만 DataStream API에서는 **"함수 그 자체"를 넘겨야** 하므로, 이 개념이 중요해집니다.

In [9]:
# 우리가 익숙한 방식: 함수를 정의하고 호출한다
def add_tax(price):
    return price * 1.1

# 함수를 "호출"하면 → 결과값(숫자)을 돌려받음
result = add_tax(10000)
print(result)        # 11000.0
print(type(result))  # <class 'float'>

print()

# 함수를 호출하지 않고(괄호 없이) 변수에 담으면?
my_func = add_tax    # 괄호가 없다! 호출이 아님!

print(my_func)             # <function add_tax at 0x...> ← 함수 "객체" 자체
print(type(my_func))       # <class 'function'>

# 이 변수를 통해 나중에 호출할 수 있음
print(my_func(10000))      # 11000.0

11000.0
<class 'float'>

<function add_tax at 0x70f01526b9c0>
<class 'function'>
11000.0


**핵심 포인트:**

- `add_tax(10000)` → 함수를 **실행**한다. 결과는 숫자 `11000.0`
- `add_tax` (괄호 없음) → 함수 **그 자체**를 가리킨다. 마치 숫자 `42`처럼 변수에 담을 수 있는 "값"

숫자, 문자열처럼 함수도 변수에 담고, 리스트에 넣고, **다른 함수의 매개변수로 전달**할 수 있습니다.
이것을 프로그래밍 용어로 **"일급 객체(First-class Object)"**라고 합니다.

### 그래서 "고차함수"란 뭔가?

정의는 간단합니다.

> **고차함수(Higher-Order Function)**: 함수를 매개변수로 받거나, 함수를 결과로 돌려주는 함수

DataFrame API에 비유하면 이렇습니다.

| DataFrame API | 고차함수 |
|---|---|
| `df.filter(col("price") > 20000)` | `filter(is_expensive, data)` |
| filter에 **조건 표현식**을 넘김 | filter에 **판별 함수**를 넘김 |
| "price가 20000보다 큰 거 골라줘" | "이 함수가 True를 뱉는 것만 골라줘" |

DataFrame API에서 `.filter()` 안에 **표현식**을 넣던 것처럼,
DataStream API에서는 `.filter()` 안에 **함수**를 넣는 것입니다.
틀(frame)은 같고, 안에 채우는 재료만 다릅니다.

### 직접 만들어 보기: 고차함수의 구조

이해를 위해, 아주 단순한 고차함수를 직접 만들어 보겠습니다.

In [10]:
# 고차함수: "어떤 함수"를 받아서, 리스트의 모든 요소에 적용한다
def apply_to_all(func, data):
    """
    func: 각 요소에 적용할 함수 (매개변수로 전달받음)
    data: 처리할 데이터 리스트
    """
    results = []
    for item in data:
        results.append(func(item))  # 전달받은 함수를 호출!
    return results

In [11]:
# apply_to_all 활용 예제
prices = [15000, 23000, 8000, 42000, 31000]

# 사용법 1: 세금 포함 가격 계산
def add_tax(x):
    return x * 1.1

result1 = apply_to_all(add_tax, prices)
print(f"세금 포함: {result1}")

# 사용법 2: 천 단위 반올림
def round_to_thousand(x):
    return round(x, -4)   # -4 : 천 단위 나타냄(-3은 백 단위)

result2 = apply_to_all(round_to_thousand, prices)
print(f"천단위 반올림: {result2}")

# 사용법 3: 가격 등급 라벨 붙이기
def price_label(x):
    return "고가" if x >= 30000 else "일반"

result3 = apply_to_all(price_label, prices)
print(f"가격 등급: {result3}")

세금 포함: [16500.0, 25300.000000000004, 8800.0, 46200.00000000001, 34100.0]
천단위 반올림: [20000, 20000, 10000, 40000, 30000]
가격 등급: ['일반', '일반', '일반', '고가', '고가']


**같은 `apply_to_all` 함수**인데, **넘겨주는 함수만 바꿨을 뿐** 전혀 다른 결과가 나옵니다.
이것이 고차함수의 힘입니다.

> 사실 `apply_to_all`은 Python 내장 함수 `map()`과 동일한 동작입니다. 뒤에서 바로 다룹니다.

### 왜 고차함수가 편한가? — 고차함수 없는 코드 vs 있는 코드

"그냥 for문 쓰면 되는 거 아닌가?" 라는 의문이 들 수 있습니다.
맞는 말입니다. 실제로 고차함수 없이도 모든 것을 구현할 수 있습니다.
하지만 비교해 보면 차이가 확실합니다.

**시나리오: 매출 데이터를 여러 가지로 가공해야 하는 상황**

이 데이터에 대해 세 가지 작업을 해야 합니다.
1. 10% 세금을 붙인 가격 계산
2. 2만 원 이상인 것만 추출
3. 전체 합계 계산

In [12]:
# ❌ 고차함수 없이: for문을 매번 작성
sales = [15000, 23000, 8000, 42000, 31000]

# 작업 1: 세금 포함 가격
taxed = []
for s in sales:
    taxed.append(s * 1.1)
print(f"세금 포함: {taxed}")

# 작업 2: 2만 원 이상 필터링
expensive = []
for s in sales:
    if s >= 20000:
        expensive.append(s)
print(f"2만원 이상: {expensive}")

# 작업 3: 합계
total = 0
for s in sales:
    total = total + s
print(f"합계: {total}")

세금 포함: [16500.0, 25300.000000000004, 8800.0, 46200.00000000001, 34100.0]
2만원 이상: [23000, 42000, 31000]
합계: 119000


In [13]:
# ✅ 고차함수 사용: 의도가 드러나는 코드
from functools import reduce

sales = [15000, 23000, 8000, 42000, 31000]

# 작업 1: 세금 포함 가격 → map (1대1 변환)
taxed = list(map(lambda s: s * 1.1, sales))
print(f"세금 포함: {taxed}")

# 작업 2: 2만 원 이상 필터링 → filter (조건 선택)
expensive = list(filter(lambda s: s >= 20000, sales))
print(f"2만원 이상: {expensive}")

# 작업 3: 합계 → reduce (집계)
total = reduce(lambda acc, s: acc + s, sales)
print(f"합계: {total}")

세금 포함: [16500.0, 25300.000000000004, 8800.0, 46200.00000000001, 34100.0]
2만원 이상: [23000, 42000, 31000]
합계: 119000


결과는 완전히 동일합니다. 하지만 코드를 읽는 경험이 다릅니다.

| 비교 항목 | for문 반복 | 고차함수 |
|---|---|---|
| 의도 파악 | 코드 내부를 읽어야 알 수 있음 | `map`=변환, `filter`=선택, `reduce`=집계 → **함수 이름만으로 파악** |
| 코드 양 | 각 작업마다 3~4줄 | 각 작업마다 1줄 |
| 반복 구조 | `for`, 빈 리스트, `append` 패턴 반복 | 반복 로직은 고차함수 내부에 숨겨짐 |
| 실수 가능성 | 변수명 오타, append 누락 등 | "무엇을 할지"만 정의하므로 실수가 줄어듦 |

그리고 이 고차함수 버전을 다시 한번 보면, DataFrame API와 구조가 닮았다는 것을 느끼실 수 있습니다.

```python
# 고차함수 (Python)
list(map(lambda s: s * 1.1, sales))
list(filter(lambda s: s >= 20000, sales))

# DataFrame API (Spark) — 전달하는 것만 "함수" 대신 "표현식"
df.withColumn("taxed", col("price") * 1.1)
df.filter(col("price") >= 20000)
```

`map`/`filter`라는 **같은 이름의 틀**에, DataFrame은 **표현식**을,
DataStream은 **함수**를 넣는 것뿐입니다.

### 잠깐, `lambda`는 뭔가요?

위 코드에서 갑자기 `lambda`가 등장했습니다. 어렵게 생각할 필요 없습니다.

> **`lambda`는 이름 없는 1회용 함수입니다.**
> "이름 붙일 정도는 아닌 간단한 함수"를 그 자리에서 바로 만들 때 사용합니다.

In [14]:
# 이 두 코드는 완전히 동일합니다

# 방법 1: 일반 함수로 정의
def add_tax(s):
    return s * 1.1

result1 = list(map(add_tax, sales))

# 방법 2: lambda로 즉석 정의
result2 = list(map(lambda s: s * 1.1, sales))

# result1과 result2는 같은 결과
print(f"일반 함수: {result1}")
print(f"lambda:    {result2}")
print(f"동일한가?  {result1 == result2}")

일반 함수: [16500.0, 25300.000000000004, 8800.0, 46200.00000000001, 34100.0]
lambda:    [16500.0, 25300.000000000004, 8800.0, 46200.00000000001, 34100.0]
동일한가?  True


`lambda s: s * 1.1`은 "매개변수 `s`를 받아서 `s * 1.1`을 돌려주는 함수"입니다.
`def`로 이름을 붙이기엔 너무 단순할 때 편리합니다.
고차함수와 자주 함께 쓰이니 눈에 익혀두시면 좋습니다.

### 고차함수가 Flink에서 왜 중요한가?

이제 핵심입니다. Flink의 DataStream API는 이런 식으로 동작합니다.

> 데이터 스트림 → `map`(변환) → `filter`(선택) → `key_by`(그룹핑) → `reduce`(집계) → 결과

DataFrame API에서 `.select().filter().groupBy().agg()`로 체이닝하던 것을 기억하시나요?
DataStream API도 똑같이 체이닝합니다. 다만 각 단계에 **표현식 대신 함수**를 넣는 것뿐입니다.

즉, 고차함수는 **데이터가 흐르는 파이프라인의 각 단계에서 "무엇을 할지"를 정의하는 도구**입니다.

Python의 기본 기능과 `functools`를 이용해, 하나씩 직접 실행 가능한 예제로 살펴보겠습니다.

### 2.1 `map`: 1대1 변환 (Transformation)

데이터를 하나 받아서, 다른 형태로 하나 배출합니다.

- **Spark SQL 비유**: `SELECT` 절에서 컬럼 값을 변경하는 것 (예: `SELECT price * 1.1 ...`)

In [15]:
# map 예제: 리스트의 모든 요소를 제곱
data = [1, 2, 3, 4, 5]

def square(x):
    return x * x

# map(함수, 데이터) → 모든 요소에 함수 적용
mapped_data = list(map(square, data))

print(f"원본: {data}")
print(f"결과: {mapped_data}")
# Flink에서의 활용: 들어온 메시지(String)를 JSON으로 파싱하거나, 필드 값을 수정할 때 사용

원본: [1, 2, 3, 4, 5]
결과: [1, 4, 9, 16, 25]


In [16]:
mapped_data = map(square, data)
print(mapped_data)

### 2.2 `filter`: 조건에 따른 선택 (Selection)

데이터를 하나 받아서, 조건이 `True`면 남기고 `False`면 버립니다.

- **Spark SQL 비유**: `WHERE` 절과 완전히 동일합니다 (예: `WHERE age > 20`)

In [17]:
# filter 예제: 짝수만 선택
data = [1, 2, 3, 4, 5, 6]

def is_even(x):
    return x % 2 == 0

# filter(함수, 데이터) → True인 것만 남김
filtered_data = list(filter(is_even, data))

print(f"원본: {data}")
print(f"결과: {filtered_data}")
# Flink에서의 활용: 불필요한 로그를 제외하거나, 특정 에러 코드만 골라낼 때 사용

원본: [1, 2, 3, 4, 5, 6]
결과: [2, 4, 6]


### 2.3 `reduce`: 집계 (Aggregation)

여러 개의 데이터를 순차적으로 합쳐서 하나의 결과로 만듭니다.

- **Spark SQL 비유**: `GROUP BY` 후 사용하는 `SUM`, `MAX`, `MIN` 등의 집계 함수

`reduce`는 두 개의 인자를 받는 함수가 필요합니다.
`(누적값, 현재값) -> 새로운 누적값` 형태로 작동합니다.

In [18]:
from functools import reduce

# reduce 예제: 리스트의 합계
data = [1, 2, 3, 4, 5]

def add(acc, current):
    print(f"  누적값: {acc}, 현재값: {current} -> 합계: {acc + current}")
    return acc + current

# reduce(함수, 데이터) → 하나의 값으로 줄임
result = reduce(add, data)
print(f"최종 결과: {result}")
# Flink에서의 활용: key_by(그룹핑) 이후에 각 그룹별 합계나 최대값을 누적할 때 사용

  누적값: 1, 현재값: 2 -> 합계: 3
  누적값: 3, 현재값: 3 -> 합계: 6
  누적값: 6, 현재값: 4 -> 합계: 10
  누적값: 10, 현재값: 5 -> 합계: 15
최종 결과: 15


In [19]:
from functools import reduce

# reduce 예제: 리스트의 합계
data = [1, 2, 3, 4, 5]

def add(acc, current):
    print(f"  누적값: {acc}, 현재값: {current} -> 합계: {acc * current}")
    return acc * current

# reduce(함수, 데이터) → 하나의 값으로 줄임
result = reduce(add, data)
print(f"최종 결과: {result}")
# Flink에서의 활용: key_by(그룹핑) 이후에 각 그룹별 합계나 최대값을 누적할 때 사용

  누적값: 1, 현재값: 2 -> 합계: 2
  누적값: 2, 현재값: 3 -> 합계: 6
  누적값: 6, 현재값: 4 -> 합계: 24
  누적값: 24, 현재값: 5 -> 합계: 120
최종 결과: 120


### 2.4 `flatMap`: 1대다 변환 (Flattening)

이것이 가장 헷갈릴 수 있는 개념입니다.
`map`과 비슷하지만, **결과가 리스트가 아니라 요소들의 나열(Stream)**로 펼쳐집니다.
1개의 입력이 0개, 1개, 혹은 여러 개의 결과가 될 수 있습니다.

- **Spark SQL 비유**: `EXPLODE` 함수와 같습니다. 배열을 행(Row)으로 쪼갤 때 사용합니다.

In [20]:
# flatMap 개념 예제: map vs flatMap 비교
sentences = ["Hello World", "Flink Python"]

# 함수 정의: 문장을 단어 리스트로 쪼갬
def split_sentence(sentence):
    return sentence.split()

# 일반 map을 쓰면? → 리스트 안에 리스트가 생김 (2차원)
mapped = list(map(split_sentence, sentences))
print(f"map 결과   : {mapped}")

# flatMap 개념 적용: 리스트를 벗겨내고 납작하게(Flat) 만듦
flat_mapped = [word for sentence in sentences for word in split_sentence(sentence)]
print(f"flatMap 결과: {flat_mapped}")
# Flink에서의 활용: WordCount에서 문장 한 줄을 받아서 여러 개의 단어로 쪼갤 때 필수

map 결과   : [['Hello', 'World'], ['Flink', 'Python']]
flatMap 결과: ['Hello', 'World', 'Flink', 'Python']


### Flink 코드 다시 보기

이제 앞서 보신 Flink 코드가 SQL로 어떻게 해석되는지 보이실 겁니다.

| Flink DataStream | SQL 대응 | 설명 |
|---|---|---|
| `ds.flat_map(split)` | `EXPLODE(SPLIT(line, ' '))` | 문장을 단어로 쪼개서 행으로 만듦 |
| `.map(lambda i: (i, 1))` | `SELECT word, 1` | 단어 옆에 숫자 1을 붙임 |
| `.key_by(lambda i: i[0])` | `GROUP BY word` | 단어 기준으로 모음 |
| `.reduce(lambda i, j: ...)` | `SUM(count)` | 1들을 계속 더함 |

**Spark SQL로 표현한다면:**

```sql
SELECT word, SUM(1)
FROM (
    SELECT EXPLODE(SPLIT(line, ' ')) as word
    FROM source_table
)
GROUP BY word
```

이 흐름을 이해하시면 Flink의 DataStream API 작성이 훨씬 수월해지실 겁니다.

### 📝 FAQ

**Q. list comprehension이 더 Pythonic 아닌가요?**

> 맞습니다. 순수 Python에서는 `[s * 1.1 for s in sales]`가 더 자연스럽습니다.
> 하지만 Flink DataStream API에서는 `ds.map(lambda s: s * 1.1)` 형태를
> 사용해야 하므로, `map`/`filter` 패턴에 익숙해지는 것이 목적입니다.

**Q. `reduce`는 Python에서 잘 안 쓰는데 왜 배우나요?**

> Python에서는 `sum()`, `max()` 같은 내장 함수가 있어서 `reduce`를 직접 쓸 일이 적습니다.
> 하지만 Flink의 `key_by().reduce()`는 **스트림에서 실시간 집계**를 수행하는 핵심 패턴입니다.
> 데이터가 무한히 흐르는 상황에서는 "미리 모아서 sum()"을 쓸 수 없기 때문에,
> 하나씩 들어올 때마다 누적하는 `reduce` 패턴이 필수적입니다.

---

### 🏋️ 실습

아래 매출 데이터에서 `map`, `filter`, `reduce`를 조합하여
**2만 원 이상인 매출에 10% 세금을 붙인 총합**을 구해 보세요.

```python
from functools import reduce
sales = [15000, 23000, 8000, 42000, 31000]

# 1. filter: 2만 원 이상만 선택
# 2. map: 10% 세금 적용
# 3. reduce: 총합 계산
# 예상 결과: (23000 + 42000 + 31000) * 1.1 = 105600.0
```

In [21]:
from functools import reduce
sales = [15000, 23000, 8000, 42000, 31000]

# 1. filter: 2만 원 이상만 선택
filtered = list(filter(lambda x : x >= 20000, sales))
print(filtered)

# 2. map: 10% 세금 적용
taxed = map(lambda x : x * 1.1, filtered)
# print(list(taxed))

# 3. reduce: 총합 계산
# acc : 누적값, cur : 현재값
# reduce(lambda acc, cur:acc + cur, taxed)
total = reduce(lambda x, y : x + y , taxed)
print(total)


[23000, 42000, 31000]
105600.00000000001


In [22]:
from functools import reduce
sales = [15000, 23000, 8000, 42000, 31000]

# 1. filter: 2만 원 이상만 선택
# 함수 사용 ver
def is_high(s):
    return s >= 20000

filtered = list(filter(is_high, sales))
# filtered

# 2. map: 10% 세금 적용
def tax(s):
    return s * 1.1

taxed = list(map(tax, filtered))
# taxed

# 3. reduce: 총합 계산
# acc : 누적값, cur : 현재값
result = reduce(lambda acc, cur:acc + cur, taxed)
result

105600.00000000001

In [23]:
[s for s in sales]

[15000, 23000, 8000, 42000, 31000]

In [24]:
[s for s in sales if s >= 20000]

[23000, 42000, 31000]

In [25]:
[round(s * 1.1) for s in sales if s >= 20000]

[25300, 46200, 34100]

In [26]:
sum(round(s * 1.1) for s in sales if s >= 20000) 

105600

---
## 3. DataStream API

DataStream API는 Flink의 저수준 API로, 스트림 처리의 핵심 구성 요소인
**상태(State)**와 **시간(Time)**에 대한 세밀한 제어가 가능합니다.

| 구분 | Spark Structured Streaming | Apache Flink |
|------|---------------------------|--------------|
| 처리 모델 | 마이크로 배치 (micro-batch) | 진정한 스트림 (true streaming) |
| 지연시간 | 수백 ms ~ 수 초 | 수 ms ~ 수십 ms |
| 상태 관리 | Checkpoint 기반 | 내장 State Backend |
| API | DataFrame/Dataset 중심 | DataStream/Table API |

### 3.1 StreamExecutionEnvironment

Spark에서 `SparkSession`을 먼저 생성하듯, Flink에서는 **`StreamExecutionEnvironment`**를
먼저 생성합니다. 이것이 Flink 프로그램의 진입점입니다.

**실행 모드:**
- `RuntimeExecutionMode.STREAMING`: 실시간 스트림 처리 (기본값)
- `RuntimeExecutionMode.BATCH`: 배치 처리 (유한한 데이터셋)

Spark에서 `trigger(once=True)`로 배치처럼 실행하는 것과 비슷하게,
Flink도 하나의 API로 배치와 스트림을 모두 처리할 수 있습니다.

In [27]:
from pyflink.datastream import StreamExecutionEnvironment

In [28]:
# 실행 환경 생성 (Spark의 SparkSession과 유사)
env = StreamExecutionEnvironment.get_execution_environment()

# 병렬성 설정 (로컬 테스트에서는 1로 설정)
env.set_parallelism(1)

print("StreamExecutionEnvironment 생성 완료")
print(f"병렬성: {env.get_parallelism()}")

StreamExecutionEnvironment 생성 완료
병렬성: 1


### 3.2 Word Count — DataStream API의 "Hello World"

Word Count는 분산 처리 시스템의 대표적인 예제입니다.
이 예제를 통해 `flat_map`, `map`, `key_by`, `reduce` 연산을 학습합니다.

In [29]:
# 예제 데이터 (셰익스피어의 햄릿 일부)
word_count_data = [
    "To be, or not to be,--that is the question:--",
    "Whether 'tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune",
    "Or to take arms against a sea of troubles,"
]

In [30]:
from pyflink.datastream import RuntimeExecutionMode
from pyflink.common import Types


In [31]:
# Word Count 구현
env = StreamExecutionEnvironment.get_execution_environment()
env.set_runtime_mode(RuntimeExecutionMode.BATCH)
env.set_parallelism(1)

# 소스: 컬렉션에서 데이터 읽기
ds = env.from_collection(word_count_data)

# 단어 분리 함수
# split : 문자열 받아서 단어로 쪼개서 리스트로 반환
# yield : 한문장 한문장 메모리에 올려서 사용 -> 전체 데이터를 한번에 메모리 저장하지 않고 필요한 순간에 하나씩 생성해서 사용
def split(line):
    yield from line.split()

# 변환 파이프라인:
# 1. flat_map: 각 라인을 단어로 분리
# 2. map: 각 단어를 (단어, 1) 튜플로 변환
# 3. key_by: 단어별로 그룹화
# 4. reduce: 같은 단어의 카운트를 합산
result = ds.flat_map(split) \
           .map(lambda word: (word, 1),
                output_type=Types.TUPLE([Types.STRING(), Types.INT()])) \
           .key_by(lambda x: x[0]) \
           .reduce(lambda a, b: (a[0], a[1] + b[1]))

result.print()
env.execute("Word Count Example")

(a,1)
(Or,1)
(To,1)
(in,1)
(is,1)
(of,2)
(or,1)
(to,3)
(The,1)
(and,1)
(be,,1)
(not,1)
(sea,1)
(the,2)
('tis,1)
(arms,1)
(mind,1)
(take,1)
(arrows,1)
(nobler,1)
(slings,1)
(suffer,1)
(Whether,1)
(against,1)
(fortune,1)
(be,--that,1)
(troubles,,1)
(outrageous,1)
(question:--,1)


**단계별 데이터 흐름 분석:**

| 단계 | 연산 | 입력 | 출력 |
|------|------|------|------|
| 1 | `flat_map(split)` | `"To be, or not to be"` | `"To"`, `"be,"`, `"or"`, `"not"`, `"to"`, `"be"` |
| 2 | `map(lambda i: (i, 1))` | `"To"` | `("To", 1)` |
| 3 | `key_by(lambda i: i[0])` | `("to", 1), ("to", 1)` | 같은 키끼리 그룹화 |
| 4 | `reduce(...)` | `("to", 1), ("to", 1)` | `("to", 2)` |

**타입 힌트의 중요성:**
Python은 동적 타입 언어라서, Flink에게 출력 타입을 명시해줘야 합니다.
`output_type=Types.TUPLE([Types.STRING(), Types.INT()])`은
"출력이 (문자열, 정수) 튜플"임을 알려줍니다.
Spark에서는 스키마 추론이 자동으로 되는 경우가 많지만,
PyFlink에서는 명시적으로 지정하는 것이 좋습니다.

#### Flink 클러스터에 Job 제출하기

위 코드는 **로컬 미니 클러스터**에서 실행됩니다.
Flink UI(http://localhost:8081)에서 Job을 확인하려면 `flink run` 명령어로
클러스터에 제출해야 합니다.

```bash
# 1. Job 파일을 작성한 뒤
flink run -py /opt/flink/src/wordcount_job.py
```

**PyFlink의 특징**: Java와 달리 `create_remote_execution_environment()`가 없어서,
노트북에서 직접 원격 클러스터에 연결할 수 없습니다.
대신 `.py` 파일로 저장 후 `flink run` 명령으로 제출합니다.

<!-- IMAGE: Flink Web UI에서 Job 실행 상태 스크린샷 -->

### 3.3 기본 연산: map, filter, key_by, sum

DataStream API의 핵심 변환 연산들을 살펴봅니다.

| Spark RDD | Flink DataStream | 설명 |
|-----------|------------------|------|
| `map()` | `map()` | 1:1 변환 |
| `flatMap()` | `flat_map()` | 1:N 변환 |
| `filter()` | `filter()` | 필터링 |
| `groupByKey()` | `key_by()` | 키 기준 그룹화 |
| `reduceByKey()` | `key_by().reduce()` | 키별 집계 |

In [32]:
# 예제 데이터: (id, JSON 문자열)
sample_data = [
    (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
    (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
    (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
    (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
]

In [33]:
import json


In [34]:
# map 연산: 각 레코드의 tel 값을 1 증가
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

# 각 데이터는 data.id, data.info 구조가 됨
ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

def update_tel(data):
    json_data = json.loads(data.info)
    json_data['tel'] += 1
    return data.id, json.dumps(json_data)

ds.map(update_tel).print()
env.execute("Map Example")

(1, '{"name": "Flink", "tel": 124, "addr": {"country": "Germany", "city": "Berlin"}}')
(2, '{"name": "hello", "tel": 136, "addr": {"country": "China", "city": "Shanghai"}}')
(3, '{"name": "world", "tel": 125, "addr": {"country": "USA", "city": "NewYork"}}')
(4, '{"name": "PyFlink", "tel": 33, "addr": {"country": "China", "city": "Hangzhou"}}')


In [35]:
# filter 연산: 조건에 맞는 레코드만 선택
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

def update_tel(data):
    json_data = json.loads(data.info)
    json_data['tel'] += 1
    return data.id, json.dumps(json_data)

# id == 1인 것만 필터링 후 map 적용
ds.filter(lambda data: data.id == 1).map(update_tel).print()
env.execute("Filter Example")

(1, '{"name": "Flink", "tel": 124, "addr": {"country": "Germany", "city": "Berlin"}}')


In [36]:
# key_by + sum: 국가별 tel 합계
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

# (country, tel) 형태로 변환 후 국가별 합계
ds.map(lambda data: (json.loads(data.info)['addr']['country'],
                     json.loads(data.info)['tel'])) \
  .key_by(lambda x: x[0]) \
  .sum(1) \
  .print()
env.execute("KeyBy Sum Example")

('Germany', 123)
('China', 135)
('USA', 124)
('China', 167)


In [37]:
# JSON 처리 파이프라인: 파싱 → 필터 → 출력
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

def parse_json(data):
    json_data = json.loads(data.info)
    json_data['tel'] += 1
    return data.id, json_data

def filter_by_country(data):
    return "China" in data[1]['addr']['country']

# China 데이터만 필터링
ds.map(parse_json).filter(filter_by_country).print()
env.execute("JSON Processing Example")

(2, {'name': 'hello', 'tel': 136, 'addr': {'country': 'China', 'city': 'Shanghai'}})
(4, {'name': 'PyFlink', 'tel': 33, 'addr': {'country': 'China', 'city': 'Hangzhou'}})


### 3.4 상태 관리 (State Management)

Flink의 핵심 기능 중 하나인 **상태 관리**를 살펴봅니다.

- **ValueState**: 키별로 단일 값을 저장
- **StateTtlConfig**: 상태의 TTL(Time-To-Live) 설정. 일정 시간이 지나면 자동 삭제

Spark에서는 상태를 메모리(Heap)에 저장하는 것이 기본이지만,
Flink는 **메모리** 또는 **RocksDB(Disk)**에 저장할 수 있어
대용량 상태도 효율적으로 관리할 수 있습니다.

In [38]:
from pyflink.datastream import KeyedProcessFunction
from pyflink.datastream.state import ValueStateDescriptor
from pyflink.common.typeinfo import Types
from pyflink.datastream.functions import RuntimeContext


In [39]:
# 상태 관리 예제: 사용자별 금액 합계
# SumFunction : key_by 이후에 동작해야 사용자별로 상태가 분리됨
class SumFunction(KeyedProcessFunction):
    """사용자별 금액을 누적 합산하는 함수"""

    # 상태를 저장할 변수를 선언만 해둠 -> 실제 초기화는 open()에서 함
    def __init__(self):
        self.state = None

    def open(self, runtime_context: RuntimeContext):
        # 상태 디스크립터 생성
        # ValueState : 키별로 하나의 값만 저장하는 상태
        state_descriptor = ValueStateDescriptor("sum_state", Types.FLOAT())

        # TTL 설정: 1초 후 만료
        # OnReadAndWrite : 읽거나 쓰면 TTL이 다시 1초로 리셋됨
        # disable_cleanup_in_background() : 백그라운드 정리 비활성화
        # 종합 : 1초동안 해당 사용자의 데이터가 안들어오면 그 사용자의 누적합 상태는 삭제됨
        state_ttl_config = StateTtlConfig \
            .new_builder(Time.seconds(1)) \
            .set_update_type(StateTtlConfig.UpdateType.OnReadAndWrite) \
            .disable_cleanup_in_background() \
            .build()
        state_descriptor.enable_time_to_live(state_ttl_config)

        # 상태 초기화
        # 이제 key 마다 별도의 상태 저장공간이 생김
        self.state = runtime_context.get_state(state_descriptor)

    def process_element(self, value, ctx: 'KeyedProcessFunction.Context'):
        # 현재 상태 값 조회
        # 현재 사용자(key)의 저장되 누적합 가져오기
        current = self.state.value()
        # 처음 들어온 사용자라면 0으로 시작
        if current is None:
            current = 0

        # 상태 업데이트
        current += value[1]
        self.state.update(current)

        # 결과 출력: (사용자, 누적합계)
        yield value[0], current

In [ ]:
from pyflink.datastream.state import StateTtlConfig
from pyflink.common.time import Time
from pyflink.datastream import StreamExecutionEnvironment
from pyflink.common.typeinfo import Types


In [41]:
# 상태 관리 실행
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

state_data = [
    ('Alice', 110.1),
    ('Bob', 30.2),
    ('Alice', 20.0),
    ('Bob', 53.1),
    ('Alice', 13.1),
    ('Bob', 3.1),
    ('Bob', 16.1),
    ('Alice', 20.1)
]

ds = env.from_collection(
    collection=state_data,
    type_info=Types.TUPLE([Types.STRING(), Types.FLOAT()])
)

ds.key_by(lambda value: value[0]) \
  .process(SumFunction()) \
  .print()

env.execute("State Access Demo")

('Alice', 110.0999984741211)
('Bob', 30.200000762939453)
('Alice', 130.0999984741211)
('Bob', 83.29999923706055)
('Alice', 143.19999885559082)
('Bob', 86.39999914169312)
('Bob', 102.49999952316284)
('Alice', 163.29999923706055)


### 🏋️ 실습

위 `SumFunction`을 수정해서 사용자별 **최대값(max)**을 추적하는
`MaxFunction`을 만들어 보세요.

```python
class MaxFunction(KeyedProcessFunction):
    # 힌트: self.state.value()와 value[1]을 비교하여
    # 더 큰 값을 state에 저장하면 됩니다.
    pass
```

<details>
<summary>모범 답안</summary>

```python
class MaxFunction(KeyedProcessFunction):
    def __init__(self):
        self.state = None

    def open(self, runtime_context: RuntimeContext):
        state_descriptor = ValueStateDescriptor("max_state", Types.FLOAT())
        self.state = runtime_context.get_state(state_descriptor)

    def process_element(self, value, ctx):
        current = self.state.value()
        if current is None or value[1] > current:
            current = value[1]
        self.state.update(current)
        yield value[0], current
```

</details>

In [68]:
class MaxFunction(KeyedProcessFunction):
    def __init__(self):
        self.state = None

    def open(self, runtime_context: RuntimeContext):
        state_descriptor = ValueStateDescriptor("max_state", Types.FLOAT())
        self.state = runtime_context.get_state(state_descriptor)

    def process_element(self, value, ctx):
        current = self.state.value()
        if current is None or value[1] > current:
            current = value[1]
        self.state.update(current)
        yield value[0], current

In [69]:
ds.key_by(lambda x: x[0]).process(MaxFunction())

### 3.5 윈도우 연산 (Window Operations)

스트림 처리에서 윈도우는 무한 데이터 스트림을 유한한 청크로 나누어 처리하는 핵심 개념입니다.

| 윈도우 유형 | 특징 | 사용 예시 |
|------------|------|----------|
| **Tumbling Window** | 고정 크기, 겹치지 않음 | 매 5분마다 집계 |
| **Sliding Window** | 고정 크기, 겹칠 수 있음 | 5분 윈도우를 1분마다 계산 |
| **Session Window** | 활동 간격 기반, 동적 크기 | 유저 세션별 집계 |
| **Count Window** | 개수 기반 | N개마다 집계 |

<!-- IMAGE: 윈도우 유형별 다이어그램 (Tumbling/Sliding/Session) -->

In [70]:
# 윈도우 예제용 데이터: (key, timestamp_ms)
window_data = [
    ('hi', 1), ('hi', 2), ('hi', 3), ('hi', 4),
    ('hi', 5), ('hi', 8), ('hi', 9), ('hi', 15)
]

# Timestamp Assigner: 튜플의 두 번째 요소를 타임스탬프로 사용
class MyTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp) -> int:
        return int(value[1])

In [71]:
# Tumbling Window: 5ms 크기, 겹치지 않는 고정 윈도우
from pyflink.datastream import ProcessWindowFunction
from pyflink.datastream.window import TumblingEventTimeWindows, TimeWindow

class CountWindowProcessFunction(ProcessWindowFunction[tuple, tuple, str, TimeWindow]):
    """윈도우 내 요소 개수를 세는 함수"""
    def process(self, key: str, context: ProcessWindowFunction.Context[TimeWindow],
                elements: Iterable[tuple]) -> Iterable[tuple]:
        count = len([e for e in elements])
        return [(key, context.window().start, context.window().end, count)]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    window_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

# 윈도우: [0,5), [5,10), [10,15), [15,20)
ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(TumblingEventTimeWindows.of(Time.milliseconds(5))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Tumbling Window Example")

(hi,0,5,4)
(hi,5,10,3)
(hi,15,20,1)


In [72]:
# Sliding Window: 크기 5ms, 슬라이드 2ms — 윈도우가 겹칠 수 있음
from pyflink.datastream.window import SlidingEventTimeWindows

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    window_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(SlidingEventTimeWindows.of(Time.milliseconds(5), Time.milliseconds(2))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Sliding Window Example")

(hi,-2,3,2)
(hi,0,5,4)
(hi,2,7,4)
(hi,4,9,3)
(hi,6,11,2)
(hi,8,13,2)
(hi,12,17,1)
(hi,14,19,1)


### 세션별로 나누는 이유
사용자의 활동 단위(세션)별로 의미 있는 집계를 하기 위해서

gap = 5ms

4 → 8 사이 간격 = 4ms ❌ (여기서 실제로는 4라서 세션 안 끊김)

하지만 보통 마지막 이벤트 이후 5ms 동안 이벤트가 없으면 세션 종료

즉: 이벤트 사이의 공백 시간(inactivity gap)이 일정 시간 이상이면 새로운 세션으로 판단

In [45]:
# Session Window: 5ms 간격 — 활동이 없으면 세션 종료
from pyflink.datastream.window import EventTimeSessionWindows

session_data = [
    ('hi', 1), ('hi', 2), ('hi', 3), ('hi', 4),  # 세션 1
    ('hi', 8), ('hi', 9),                          # 세션 2 (5ms 이상 간격)
    ('hi', 15)                                      # 세션 3
]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    session_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(EventTimeSessionWindows.with_gap(Time.milliseconds(5))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Session Window Example")

(hi,1,14,6)
(hi,15,20,1)


### count window
- 시간이 아니라 “개수”로 윈도우를 자름 
- ex) count_window(2)는 key별로 이벤트를 2개(여기선 'hi', 'hello') 모을 때마다 하나의 윈도우를 닫고 apply()를 실행
- “몇 개 모였냐”만 봄


In [73]:
# Count Window: 2개씩 그룹화 — 개수 기반 윈도우
from pyflink.datastream import WindowFunction
from pyflink.datastream.window import CountWindow

class SumWindowFunction(WindowFunction[tuple, tuple, str, CountWindow]):
    """윈도우 내 값의 합계를 계산"""
    def apply(self, key: str, window: CountWindow, inputs: Iterable[tuple]):
        total = sum(i[0] for i in inputs)
        return [(key, total)]

count_data = [
    (1, 'hi'), (2, 'hello'), (3, 'hi'), (4, 'hello'),
    (5, 'hi'), (6, 'hello'), (6, 'hello')
]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    count_data,
    type_info=Types.TUPLE([Types.INT(), Types.STRING()])
)

# hi: [1, 3] -> 4, [5] -> 대기 (2개가 안 채워져서)
# hello: [2, 4] -> 6, [6, 6] -> 12
ds.key_by(lambda x: x[1], key_type=Types.STRING()) \
  .count_window(2) \
  .apply(SumWindowFunction(), Types.TUPLE([Types.STRING(), Types.INT()])) \
  .print()

env.execute("Count Window Example")

(hi,4)
(hello,6)
(hello,12)


key='hi' 에 들어오는 값들

값(첫 번째 필드)만 보면: [1, 3, 5]

윈도우1: [1, 3] → 합 4 ✅ 출력

윈도우2: [5] → 아직 2개 안 됨 → 출력 안 됨(대기)

key='hello' 에 들어오는 값들

값만 보면: [2, 4, 6, 6]

윈도우1: [2, 4] → 합 6 ✅ 출력

윈도우2: [6, 6] → 합 12 ✅ 출력

출력 순서는 스트림 처리/키 도착 순서에 따라 섞일 수 있지만, 나오는 결과 값 자체는 이 3개가 맞아요.

('hi', 4)

('hello', 6)

('hello', 12)

('hi', 5)는 윈도우가 닫히지 않아서 안 나옵니다.

### 3.6 이벤트 시간과 타이머

Flink는 이벤트 시간 기반 처리를 위한 타이머 기능을 제공합니다.
타이머를 등록하면 특정 시간이 지난 후에 콜백 함수가 호출됩니다.

In [74]:
# 이벤트 타이머 예제: 각 이벤트 처리 후 2초 뒤에 합계 출력
class TimerSumFunction(KeyedProcessFunction):
    """타이머를 사용한 지연 출력 함수"""

    def __init__(self):
        self.state = None

    def open(self, runtime_context: RuntimeContext):
        state_descriptor = ValueStateDescriptor("state", Types.FLOAT())
        state_ttl_config = StateTtlConfig \
            .new_builder(Time.seconds(1)) \
            .set_update_type(StateTtlConfig.UpdateType.OnReadAndWrite) \
            .disable_cleanup_in_background() \
            .build()
        state_descriptor.enable_time_to_live(state_ttl_config)
        self.state = runtime_context.get_state(state_descriptor)

    def process_element(self, value, ctx: 'KeyedProcessFunction.Context'):
        current = self.state.value() or 0
        current += value[2]
        self.state.update(current)
        # 2초 후에 타이머 발동 등록
        ctx.timer_service().register_event_time_timer(ctx.timestamp() + 2000)

    def on_timer(self, timestamp: int, ctx: 'KeyedProcessFunction.OnTimerContext'):
        """타이머 발동 시 호출"""
        yield ctx.get_current_key(), self.state.value()

class EventTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp: int) -> int:
        return int(value[0])

In [75]:
# 이벤트 타이머 실행
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

timer_data = [
    (1000, 'Alice', 110.1),
    (4000, 'Bob', 30.2),
    (3000, 'Alice', 20.0),
    (2000, 'Bob', 53.1),
    (5000, 'Alice', 13.1),
    (3000, 'Bob', 3.1),
    (7000, 'Bob', 16.1),
    (10000, 'Alice', 20.1)
]

ds = env.from_collection(
    collection=timer_data,
    type_info=Types.TUPLE([Types.LONG(), Types.STRING(), Types.FLOAT()])
)

ds.assign_timestamps_and_watermarks(
    WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(2))
                     .with_timestamp_assigner(EventTimestampAssigner())
) \
  .key_by(lambda value: value[1]) \
  .process(TimerSumFunction()) \
  .print()

env.execute("Event Timer Example")

('Alice', 163.29999923706055)
('Bob', 102.49999952316284)
('Alice', 163.29999923706055)
('Bob', 102.49999952316284)
('Bob', 102.49999952316284)
('Alice', 163.29999923706055)
('Bob', 102.49999952316284)
('Alice', 163.29999923706055)


### 3.7 Source/Sink 패턴 참조

**소스(Source) 유형:**

| 소스 유형 | 설명 | Spark 대응 |
|----------|------|-----------|
| `from_collection()` | 메모리 리스트에서 읽기 (테스트용) | `spark.createDataFrame()` |
| `FileSource` | 파일에서 읽기 | `spark.readStream.format("text")` |
| `KafkaSource` | Kafka 토픽에서 읽기 | `spark.readStream.format("kafka")` |

**싱크(Sink) 유형:**

| 싱크 유형 | 설명 | Spark 대응 |
|----------|------|-----------|
| `.print()` | 콘솔 출력 (디버깅용) | `df.show()` |
| `FileSink` | 파일로 쓰기 | `df.writeStream.format("text")` |
| `KafkaSink` | Kafka 토픽으로 쓰기 | `df.writeStream.format("kafka")` |

**`env.execute()` — Lazy Evaluation:**

Flink는 Spark처럼 **지연 실행** 방식입니다.
변환을 정의해도 바로 실행되지 않고, 최종적으로 `execute()`를 호출해야 실제로 작업이 수행됩니다.
모든 소스, 변환, 싱크를 정의한 후 마지막에 `env.execute()`를 호출하는 것이 기본 패턴입니다.

### 📝 FAQ

**Q. `env.execute()`는 왜 마지막에 호출하나요?**

> Flink는 Spark처럼 지연 실행(Lazy Evaluation) 방식입니다.
> `map()`, `filter()` 등은 실행 계획만 등록할 뿐, 실제 데이터 처리는 하지 않습니다.
> `execute()`를 호출하는 시점에 전체 파이프라인이 최적화되어 한 번에 실행됩니다.

**Q. `output_type`을 꼭 지정해야 하나요?**

> Python은 동적 타입 언어라서, Flink(Java 기반)가 출력 타입을 자동 추론하기 어렵습니다.
> 특히 `lambda`를 사용할 때는 반드시 `output_type`을 지정해야 합니다.
> 지정하지 않으면 직렬화 오류가 발생할 수 있습니다.

**Q. State TTL은 왜 설정하나요?**

> 스트림 처리는 24시간 365일 실행됩니다. 상태를 무한히 쌓으면 메모리가 고갈됩니다.
> TTL을 설정하면 일정 시간이 지난 상태가 자동으로 삭제되어 메모리를 효율적으로 관리할 수 있습니다.
> 예: "최근 1시간 동안의 사용자 활동만 추적"

**Q. Window가 닫히는 시점은?**

> Watermark가 윈도우 끝 시간을 넘어서면 윈도우가 닫힙니다.
> 예: Tumbling Window [0, 5)에서 Watermark가 5 이상이 되면 윈도우가 닫히고 결과가 출력됩니다.
> Flink는 Spark와 달리 배치 경계가 아닌 Watermark 도달 즉시 윈도우를 닫습니다.

**Q. Count Window에서 남은 데이터는 어떻게 되나요?**

> Count Window는 지정된 개수가 채워져야 결과가 출력됩니다.
> 예: 2개씩 그룹화할 때 3개의 데이터가 들어오면, 2개는 처리되고 1개는 대기합니다.
> 스트림이 종료되면 남은 데이터는 처리되지 않습니다.

### ✅ 퀴즈

**Q1.** Flink DataStream API에서 `key_by()`에 대응하는 Spark 연산은?

- A) `select()`
- B) `groupByKey()`
- C) `repartition()`
- D) `broadcast()`

<details>
<summary>정답 확인</summary>

**정답: B)**

`key_by()`는 키 기준으로 데이터를 그룹화하는 연산으로, Spark의 `groupByKey()`와 대응됩니다.
Flink에서는 `key_by()` 이후에 `reduce()`, `sum()`, `process()` 등의 집계 연산을 적용합니다.

</details>

---

**Q2.** Tumbling Window와 Sliding Window의 핵심 차이점은?

- A) Tumbling은 시간 기반, Sliding은 개수 기반
- B) Tumbling은 윈도우가 겹치지 않고, Sliding은 겹칠 수 있음
- C) Tumbling은 배치용, Sliding은 스트림용
- D) Tumbling은 키가 필요하고, Sliding은 키가 불필요

<details>
<summary>정답 확인</summary>

**정답: B)**

Tumbling Window는 고정 크기의 겹치지 않는 윈도우입니다 (예: 매 5초).
Sliding Window는 고정 크기이지만 슬라이드 간격에 따라 겹칠 수 있습니다 (예: 5초 윈도우를 2초마다).

</details>

---

**Q3.** ValueState에 TTL을 설정하는 가장 큰 이유는?

- A) 보안을 위해 오래된 데이터를 삭제
- B) 처리 속도를 높이기 위해
- C) 무한히 쌓이는 상태로 인한 메모리 고갈 방지
- D) 체크포인트 크기를 줄이기 위해

<details>
<summary>정답 확인</summary>

**정답: C)**

스트림 처리는 24/7 실행되므로 상태가 무한히 쌓일 수 있습니다.
TTL을 설정하면 일정 시간이 지난 상태를 자동 삭제하여 메모리를 효율적으로 관리합니다.
(D도 부수적인 효과이지만, 핵심 이유는 메모리 관리입니다.)

</details>

**DataStream API 개념 매핑표:**

| Flink 개념 | Spark 대응 개념 | 설명 |
|-----------|----------------|------|
| `StreamExecutionEnvironment` | `SparkSession` | 실행 환경/컨텍스트 |
| `DataStream` | `RDD` / `DStream` | 데이터 스트림 추상화 |
| `Source` | `readStream` | 데이터 입력 |
| `Sink` | `writeStream` | 데이터 출력 |
| `key_by()` | `groupByKey()` | 키 기준 파티셔닝 |
| `WatermarkStrategy` | `withWatermark()` | 이벤트 시간 처리 |

---
## 4. Table API

앞서 배운 DataStream API가 Spark의 RDD API와 유사했다면,
**Table API**는 Spark의 **DataFrame API / SQL**과 매우 유사합니다.
더 선언적이고 SQL 친화적인 방식으로 데이터를 처리할 수 있습니다.

| 구분 | DataStream API | Table API |
|------|---------------|-----------|
| 추상화 수준 | 저수준 (Low-level) | 고수준 (High-level) |
| 비유 | Spark RDD | Spark DataFrame |
| 스키마 | 명시적 타입 힌트 필요 | 테이블 스키마 기반 |
| SQL 지원 | 불가능 | 가능 |
| 최적화 | 수동 | 자동 (Optimizer) |
| 사용 사례 | 세밀한 제어 필요 시 | 분석/ETL 작업 |

**Spark 경험이 있다면:** DataFrame API를 쓰듯이 Table API를 사용하면 됩니다!

### 4.1 TableEnvironment

DataStream API에서 `StreamExecutionEnvironment`를 사용했다면,
Table API에서는 **`TableEnvironment`**를 사용합니다.

```python
# Spark 방식
spark = SparkSession.builder \
    .config("spark.default.parallelism", "1") \
    .getOrCreate()

# Flink Table API 방식
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")
```

**실행 모드 선택:**
- `EnvironmentSettings.in_streaming_mode()`: 스트리밍 모드
- `EnvironmentSettings.in_batch_mode()`: 배치 모드

In [76]:
# Table 환경 생성
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

print("TableEnvironment 생성 완료")

TableEnvironment 생성 완료


### 4.2 테이블 정의 + Word Count

Flink Table API에서 소스/싱크 테이블을 정의하는 방법은 두 가지가 있습니다.

**방법 1: TableDescriptor (프로그래밍 방식)**

```python
t_env.create_temporary_table(
    'source',
    TableDescriptor.for_connector('filesystem')
        .schema(Schema.new_builder()
                .column('word', DataTypes.STRING())
                .build())
        .option('path', input_path)
        .format('csv')
        .build()
)
```

**방법 2: DDL SQL 사용 (SQL 친화적)**

```python
t_env.execute_sql("""
    CREATE TABLE source (word STRING)
    WITH ('connector'='filesystem', 'format'='csv', 'path'='/path/to/input')
""")
```

Spark SQL의 `CREATE TABLE ... USING csv LOCATION ...`과 비슷하지만,
Flink는 `WITH` 절로 커넥터 옵션을 지정합니다.

In [77]:
# Table API Word Count
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

word_count_data = [
    "To be, or not to be,--that is the question:--",
    "Whether 'tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune"
]

# 소스 테이블 생성
tab = t_env.from_elements(
    map(lambda i: (i,), word_count_data),
    DataTypes.ROW([DataTypes.FIELD('line', DataTypes.STRING())])
)

# Sink 테이블 생성 (print connector)
t_env.create_temporary_table(
    'sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('word', DataTypes.STRING())
                           .column('count', DataTypes.BIGINT())
                           .build())
                   .build()
)

# UDTF: 한 행을 여러 행으로 확장 (DataStream의 flat_map과 대응)
@udtf(result_types=[DataTypes.STRING()])
def split(line: Row):
    for word in line[0].split():
        yield Row(word)

# 파이프라인: flat_map -> alias -> group_by -> count -> insert
tab.flat_map(split).alias('word') \
   .group_by(col('word')) \
   .select(col('word'), lit(1).count) \
   .execute_insert('sink') \
   .wait()

+I[To, 1]
+I[be,, 1]
+I[or, 1]
+I[not, 1]
+I[to, 1]
+I[be,--that, 1]
+I[is, 1]
+I[the, 1]
+I[question:--, 1]
+I[Whether, 1]
+I['tis, 1]
+I[nobler, 1]
+I[in, 1]
-U[the, 1]
+U[the, 2]
+I[mind, 1]
-U[to, 1]
+U[to, 2]
+I[suffer, 1]
+I[The, 1]
+I[slings, 1]
+I[and, 1]
+I[arrows, 1]
+I[of, 1]
+I[outrageous, 1]
+I[fortune, 1]


**단계별 분석:**

| 단계 | 코드 | 설명 |
|------|------|------|
| 1 | `tab.flat_map(split)` | UDTF로 각 줄을 단어들로 분리 |
| 2 | `.alias('word')` | 결과 컬럼에 'word' 이름 부여 |
| 3 | `.group_by(col('word'))` | 단어별 그룹화 (Spark의 `groupBy`와 동일) |
| 4 | `.select(col('word'), lit(1).count)` | 집계: `COUNT(1)` |
| 5 | `.execute_insert('sink')` | 결과를 sink 테이블에 삽입하고 실행 |
| 6 | `.wait()` | 로컬 실행 시 완료 대기 |

**Changelog Stream (+I, -U, +U, -D):**

출력에서 `+I`, `-U`, `+U`, `-D` 같은 접두사를 볼 수 있습니다.
이는 Flink의 **Changelog Stream** 개념입니다.

| 접두사 | 의미 | 설명 |
|--------|------|------|
| `+I` | INSERT | 새 레코드 추가 |
| `-U` | UPDATE BEFORE | 업데이트 전 값 (이전 값 취소) |
| `+U` | UPDATE AFTER | 업데이트 후 값 (새 값 삽입) |
| `-D` | DELETE | 삭제 |

스트리밍에서 `GROUP BY` 결과가 바뀔 때마다 이전 값을 취소(-U)하고
새 값을 삽입(+U)하는 방식으로 실시간 업데이트를 표현합니다.

### 4.3 기본 연산

Table API의 기본 연산들을 살펴봅니다.

| 연산 | Table API | Spark DataFrame | SQL |
|------|-----------|-----------------|-----|
| 필터 | `table.filter()` | `df.filter()` | `WHERE` |
| 제한 | `table.limit()` | `df.limit()` | `LIMIT` |
| 그룹 집계 | `table.group_by().select()` | `df.groupBy().agg()` | `GROUP BY` |
| 중복 제거 | `table.distinct()` | `df.distinct()` | `DISTINCT` |
| 조인 | `table.join()` | `df.join()` | `JOIN` |

In [78]:
# JSON 필드 추출
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

# json_value()로 JSON 필드 추출
table = table.add_columns(
    col('data').json_value('$.name', DataTypes.STRING()).alias('name'),
    col('data').json_value('$.tel', DataTypes.STRING()).alias('tel'),
    col('data').json_value('$.addr.country', DataTypes.STRING()).alias('country')
).drop_columns(col('data'))

table.execute().print()

+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| op |                   id |                           name |                            tel |                        country |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| +I |                    1 |                          Flink |                            123 |                        Germany |
| +I |                    2 |                          hello |                            135 |                          China |
| +I |                    3 |                          world |                            124 |                            USA |
| +I |                    4 |                        PyFlink |                             32 |                          China |
+----+----------------------+--------------------------------+--------------------------------+--

In [52]:
# limit
table.limit(3).execute().print()

+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| op |                   id |                           name |                            tel |                        country |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| +I |                    1 |                          Flink |                            123 |                        Germany |
| +I |                    2 |                          hello |                            135 |                          China |
| +I |                    3 |                          world |                            124 |                            USA |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
3 rows in set


In [53]:
# filter
table.filter(col('id') != 3).execute().print()

+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| op |                   id |                           name |                            tel |                        country |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
| +I |                    1 |                          Flink |                            123 |                        Germany |
| +I |                    2 |                          hello |                            135 |                          China |
| +I |                    4 |                        PyFlink |                             32 |                          China |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+
3 rows in set


In [54]:
# group_by + aggregation: 국가별 집계
table.group_by(col('country')) \
     .select(
         col('country'),
         col('id').count.alias('cnt'),
         col('tel').cast(DataTypes.BIGINT()).max.alias('max_tel')
     ) \
     .execute().print()

+----+--------------------------------+----------------------+----------------------+
| op |                        country |                  cnt |              max_tel |
+----+--------------------------------+----------------------+----------------------+
| +I |                        Germany |                    1 |                  123 |
| +I |                          China |                    1 |                  135 |
| -U |                          China |                    1 |                  135 |
| +U |                          China |                    2 |                  135 |
| +I |                            USA |                    1 |                  124 |
+----+--------------------------------+----------------------+----------------------+
5 rows in set


In [ ]:
# distinct : 중복 제거
table.select(col('country')).distinct().execute().print()

+----+--------------------------------+
| op |                        country |
+----+--------------------------------+
| +I |                        Germany |
| +I |                          China |
| +I |                            USA |
+----+--------------------------------+
3 rows in set


In [56]:
# join
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

table = table.add_columns(
    col('data').json_value('$.name', DataTypes.STRING()).alias('name'),
    col('data').json_value('$.tel', DataTypes.STRING()).alias('tel'),
    col('data').json_value('$.addr.country', DataTypes.STRING()).alias('country')
).drop_columns(col('data'))

right_table = t_env.from_elements(
    elements=[(1, 18), (2, 30), (3, 25), (4, 10)],
    schema=['id', 'age']
)

table.join(
    right_table.rename_columns(col('id').alias('r_id')),
    col('id') == col('r_id')
).execute().print()

+----+----------------------+--------------------------------+--------------------------------+--------------------------------+----------------------+----------------------+
| op |                   id |                           name |                            tel |                        country |                 r_id |                  age |
+----+----------------------+--------------------------------+--------------------------------+--------------------------------+----------------------+----------------------+
| +I |                    4 |                        PyFlink |                             32 |                          China |                    4 |                   10 |
| +I |                    1 |                          Flink |                            123 |                        Germany |                    1 |                   18 |
| +I |                    2 |                          hello |                            135 |                          Chin

### 4.4 UDF + Expression API

**Expression API** — Spark DataFrame의 컬럼 표현식과 유사합니다:

```python
from pyflink.table.expressions import col, lit

col('word')           # Spark: col("word")
lit(1)                # Spark: lit(1)
lit(1).count          # Spark: count(lit(1))
col('amount').sum     # Spark: sum("amount")
col('age') > 18       # Spark: col("age") > 18
```

**UDTF (User Defined Table Function):**

Table API에서 `flat_map`처럼 1:N 변환을 하려면 UDTF를 사용합니다.
DataStream API의 `flat_map`과 대응되는 개념입니다.

```python
# DataStream API
def split(line):
    yield from line.split()
ds.flat_map(split)

# Table API — UDTF 데코레이터 필요
@udtf(result_types=[DataTypes.STRING()])
def split(line: Row):
    for s in line[0].split():
        yield Row(s)
tab.flat_map(split)
```

### UDF
- 사용자 정의 함수(User Defined Function)
- Flink에서 기본 제공하는 함수가 아닌 사용자가 직접 만든 함수

### UDTF
- User Defined Table Function
- 사용자가 만든 여러 행(row)를 반환하는 함수
- 입력 1개 -> 출력 여러개(여러 행) 이라 table function임
- 사용 ex) 문자열을 단어로 분리(explode) / 배열을 여러 행으로 펼치기 / JSON 내부 배열 분해 등

### Expression API
- SQL을 문자열로 쓰지 않고, 파이썬 코드 형태로 표현식을 작성하는 방식
- SQL 방식
    ```python
    table.select("id, amount + 1")
    ```
- Expression API 방식
    ```python
    from pyflink.table import expressions as expr

    table.select(
        expr.col("id"),
        (expr.col("amount") + 1)
    )
    ```

In [57]:
# UDF 예제: JSON의 tel 값 증가
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}')
    ],
    schema=['id', 'data']
)

t_env.create_temporary_table(
    'sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('id', DataTypes.BIGINT())
                           .column('data', DataTypes.STRING())
                           .build())
                   .build()
)

@udf(result_type=DataTypes.STRING())
def update_tel(data):
    json_data = json.loads(data)
    json_data['tel'] += 1
    return json.dumps(json_data)

table.select(col('id'), update_tel(col('data'))) \
     .execute_insert('sink') \
     .wait()

Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
    target=lambda: self._read_inputs(elements_iterator),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 704, in _read_inputs
    for elements in elements_iterator:
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 543, in __next__
    return self._next()
           ^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 969, in _next
    raise self
grpc._channel._MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = Sta

18> +I[1, {"name": "Flink", "tel": 124, "addr": {"country": "Germany", "city": "Berlin"}}]
18> +I[2, {"name": "hello", "tel": 136, "addr": {"country": "China", "city": "Shanghai"}}]


### 4.5 SQL 연산

Spark SQL 경험이 있으면 바로 사용할 수 있습니다.
Table API와 SQL을 자유롭게 혼합할 수 있습니다.

In [ ]:
# SQL 쿼리 사용
# 1. Table API / SQL 실행하는 실행 환경을 스트리밍 모드로 만들겠다
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

# 2. 테이블 생성
# from_elements() : 리스트 -> Flink Table 객체로 변환
table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

# sql_query() : SQL 문자열을 실행해서 새로운 table 반환
# execute() : 실제 실행
# print() : 결과 출력
t_env.sql_query(f"SELECT * FROM {table}").execute().print()

+----+----------------------+--------------------------------+
| op |                   id |                           data |
+----+----------------------+--------------------------------+
| +I |                    1 | {"name": "Flink", "tel": 12... |
| +I |                    2 | {"name": "hello", "tel": 13... |
| +I |                    3 | {"name": "world", "tel": 12... |
| +I |                    4 | {"name": "PyFlink", "tel": ... |
+----+----------------------+--------------------------------+
4 rows in set


In [59]:
# SQL에서 UDTF 사용 (LATERAL TABLE)
@udtf(result_types=[DataTypes.STRING(), DataTypes.INT(), DataTypes.STRING()])
def parse_data(data: str):
    json_data = json.loads(data)
    yield json_data['name'], json_data['tel'], json_data['addr']['country']

t_env.create_temporary_function('parse_data', parse_data)

t_env.execute_sql(f"""
    SELECT id, name, tel, country
    FROM {table}, LATERAL TABLE(parse_data(`data`)) t(name, tel, country)
""").print()

+----+----------------------+--------------------------------+-------------+--------------------------------+
| op |                   id |                           name |         tel |                        country |
+----+----------------------+--------------------------------+-------------+--------------------------------+
| +I |                    1 |                          Flink |         123 |                        Germany |
| +I |                    2 |                          hello |         135 |                          China |
| +I |                    3 |                          world |         124 |                            USA |
| +I |                    4 |                        PyFlink |          32 |                          China |
+----+----------------------+--------------------------------+-------------+--------------------------------+
4 rows in set


### 4.6 Table API 윈도우

In [60]:
# Table API 윈도우용 데이터 준비
window_table_data = [
    (Instant.of_epoch_milli(1000), 'Alice', 110.1),
    (Instant.of_epoch_milli(4000), 'Bob', 30.2),
    (Instant.of_epoch_milli(3000), 'Alice', 20.0),
    (Instant.of_epoch_milli(2000), 'Bob', 53.1),
    (Instant.of_epoch_milli(5000), 'Alice', 13.1),
    (Instant.of_epoch_milli(3000), 'Bob', 3.1),
    (Instant.of_epoch_milli(7000), 'Bob', 16.1),
    (Instant.of_epoch_milli(10000), 'Alice', 20.1)
]

In [61]:
# Tumble Window (Table API): 5초 고정 윈도우
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'tumble_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('total_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

table.window(Tumble.over(lit(5).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), col('price').sum, col("w").start, col("w").end) \
     .execute_insert('tumble_sink') \
     .wait()

+I[Alice, 130.1, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 86.4, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 16.1, 1970-01-01T00:00:05Z, 1970-01-01T00:00:10Z]
+I[Alice, 13.1, 1970-01-01T00:00:05Z, 1970-01-01T00:00:10Z]
+I[Alice, 20.1, 1970-01-01T00:00:10Z, 1970-01-01T00:00:15Z]


In [62]:
# Sliding Window (Table API): 크기 5초, 슬라이드 2초
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'slide_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('total_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

table.window(Slide.over(lit(5).seconds).every(lit(2).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), col('price').sum, col("w").start, col("w").end) \
     .execute_insert('slide_sink') \
     .wait()

+I[Alice, 110.1, 1969-12-31T23:59:58Z, 1970-01-01T00:00:03Z]
+I[Bob, 53.1, 1969-12-31T23:59:58Z, 1970-01-01T00:00:03Z]
+I[Alice, 130.1, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 86.399994, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 86.399994, 1970-01-01T00:00:02Z, 1970-01-01T00:00:07Z]
+I[Alice, 33.1, 1970-01-01T00:00:02Z, 1970-01-01T00:00:07Z]
+I[Alice, 13.1, 1970-01-01T00:00:04Z, 1970-01-01T00:00:09Z]
+I[Bob, 46.300003, 1970-01-01T00:00:04Z, 1970-01-01T00:00:09Z]
+I[Bob, 16.1, 1970-01-01T00:00:06Z, 1970-01-01T00:00:11Z]
+I[Alice, 20.1, 1970-01-01T00:00:06Z, 1970-01-01T00:00:11Z]
+I[Alice, 20.1, 1970-01-01T00:00:08Z, 1970-01-01T00:00:13Z]
+I[Alice, 20.1, 1970-01-01T00:00:10Z, 1970-01-01T00:00:15Z]


In [63]:
# Over Window (Table API): 행 기반 윈도우 — 이전 2행까지의 max
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'over_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('max_price', DataTypes.FLOAT())
                           .build())
                   .build()
)

table.over_window(
    Over.partition_by(col("name"))
        .order_by(col("ts"))
        .preceding(row_interval(2))
        .following(CURRENT_ROW)
        .alias('w')
) \
     .select(col('name'), col('price').max.over(col('w'))) \
     .execute_insert('over_sink') \
     .wait()

+I[Alice, 110.1]
+I[Bob, 53.1]
+I[Alice, 110.1]
+I[Bob, 53.1]
+I[Bob, 53.1]
+I[Alice, 110.1]
+I[Bob, 30.2]
+I[Alice, 20.1]


### 4.7 Multi-Sink + Pandas

**StatementSet**: 하나의 소스 데이터를 여러 싱크로 동시에 출력할 수 있습니다.

In [64]:
# Multi-Sink 예제: 하나의 소스 → 두 개의 싱크
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[(1, 'Hello'), (2, 'World'), (3, "Flink"), (4, "PyFlink")],
    schema=['id', 'data']
)

t_env.execute_sql("""
    CREATE TABLE first_sink (
        id BIGINT,
        data VARCHAR
    ) WITH ('connector' = 'print')
""")

t_env.execute_sql("""
    CREATE TABLE second_sink (
        id BIGINT,
        data VARCHAR
    ) WITH ('connector' = 'print')
""")

# StatementSet으로 여러 싱크에 동시 출력
statement_set = t_env.create_statement_set()

# Sink 1: id <= 3인 데이터
statement_set.add_insert_sql(f"INSERT INTO first_sink SELECT * FROM {table} WHERE id <= 3")

# Sink 2: 'Flink' 포함 데이터
@udf(result_type=DataTypes.BOOLEAN())
def contains_flink(data):
    return "Flink" in data

second_table = table.where(contains_flink(table.data))
statement_set.add_insert("second_sink", second_table)

statement_set.execute().wait()

Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
    target=lambda: self._read_inputs(elements_iterator),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 704, in _read_inputs
    for elements in elements_iterator:
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 543, in __next__
    return self._next()
           ^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 969, in _next
    raise self
grpc._channel._MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = Sta

13> +I[1, Hello]
13> +I[2, World]
13> +I[3, Flink]
13> +I[3, Flink]
13> +I[4, PyFlink]


In [65]:
# Pandas DataFrame ↔ Flink Table 변환
import pandas as pd
import numpy as np

t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

# Pandas DataFrame 생성
pdf = pd.DataFrame(np.random.rand(5, 2), columns=['a', 'b'])
print("원본 Pandas DataFrame:")
print(pdf)

# Pandas → Flink Table
table = t_env.from_pandas(
    pdf,
    schema=DataTypes.ROW([
        DataTypes.FIELD("a", DataTypes.DOUBLE()),
        DataTypes.FIELD("b", DataTypes.DOUBLE())
    ])
)

# Flink Table → Pandas
result_pdf = table.to_pandas()
print("\nFlink를 거쳐 다시 Pandas로:")
print(result_pdf)

원본 Pandas DataFrame:
          a         b
0  0.500287  0.244131
1  0.203126  0.875233
2  0.425922  0.124446
3  0.232521  0.393878
4  0.968117  0.677180

Flink를 거쳐 다시 Pandas로:
          a         b
0  0.500287  0.244131
1  0.203126  0.875233
2  0.425922  0.124446
3  0.232521  0.393878
4  0.968117  0.677180


In [66]:
# Pandas UDAF: 윈도우별 평균 계산
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP_LTZ(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'pandas_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('mean_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

# Pandas UDAF: Pandas의 mean() 사용
@udaf(result_type=DataTypes.FLOAT(), func_type="pandas")
def mean_udaf(v):
    return v.mean()

table.window(Tumble.over(lit(5).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), mean_udaf(col('price')), col("w").start, col("w").end) \
     .execute_insert('pandas_sink') \
     .wait()

+I[Alice, 65.05, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 28.800001, 1970-01-01T00:00:00Z, 1970-01-01T00:00:05Z]
+I[Bob, 16.1, 1970-01-01T00:00:05Z, 1970-01-01T00:00:10Z]
+I[Alice, 13.1, 1970-01-01T00:00:05Z, 1970-01-01T00:00:10Z]
+I[Alice, 20.1, 1970-01-01T00:00:10Z, 1970-01-01T00:00:15Z]


### 📝 FAQ

**Q. Table API와 SQL 중 어떤 걸 써야 하나요?**

> 둘 다 내부적으로 같은 실행 엔진을 사용하므로 성능 차이는 없습니다.
> SQL에 익숙하면 SQL을, 프로그래밍 방식을 선호하면 Table API를 사용하세요.
> 실무에서는 혼합해서 쓰는 경우가 많습니다.

**Q. Changelog Stream의 -U/+U는 뭔가요?**

> 스트리밍 `GROUP BY`에서 같은 키의 집계 결과가 변경될 때,
> 이전 결과를 취소(-U)하고 새 결과를 삽입(+U)하는 방식입니다.
> 예: `("hello", 1)` → 새 "hello" 도착 → `-U("hello", 1)`, `+U("hello", 2)`

**Q. Pandas UDAF는 느리지 않나요?**

> JVM과 Python 간 데이터 전송 오버헤드가 있지만, Apache Arrow를 사용하여
> 직렬화/역직렬화를 최적화합니다. Pandas의 벡터화 연산을 활용할 수 있어
> 행 단위 처리보다 빠른 경우도 있습니다.

**Q. `execute_insert()` vs `execute()` 차이는?**

> `execute_insert('sink')`: 결과를 특정 싱크 테이블에 삽입하고 실행합니다.
> `table.execute()`: 결과를 클라이언트로 가져와서 출력합니다 (`.print()` 등).
> 프로덕션에서는 `execute_insert()`를, 디버깅에서는 `.execute().print()`를 사용합니다.

### ✅ 퀴즈

**Q1.** Table API에서 `group_by()`에 대응하는 Spark DataFrame 메서드는?

- A) `select()`
- B) `groupBy()`
- C) `partition_by()`
- D) `distinct()`

<details>
<summary>정답 확인</summary>

**정답: B)**

Flink Table API의 `group_by()`는 Spark DataFrame의 `groupBy()`와 동일한 역할을 합니다.
둘 다 지정된 컬럼을 기준으로 데이터를 그룹화한 뒤 집계 연산을 수행합니다.

</details>

---

**Q2.** UDTF와 UDF의 핵심 차이점은?

- A) UDTF는 Python 전용, UDF는 Java 전용
- B) UDTF는 1개 입력에서 여러 행을 생성, UDF는 1개 입력에서 1개 값을 반환
- C) UDTF는 배치용, UDF는 스트리밍용
- D) UDTF는 집계 함수, UDF는 변환 함수

<details>
<summary>정답 확인</summary>

**정답: B)**

UDF(User Defined Function)는 1:1 변환으로 하나의 입력에서 하나의 값을 반환합니다.
UDTF(User Defined Table Function)는 1:N 변환으로 하나의 입력에서 여러 행을 생성합니다.
DataStream API의 `map` vs `flat_map` 관계와 동일합니다.

</details>

---

**Q3.** StatementSet의 용도로 가장 적절한 것은?

- A) 여러 소스에서 데이터를 읽기 위해
- B) 하나의 소스 데이터를 여러 싱크에 동시 출력하기 위해
- C) 트랜잭션 처리를 위해
- D) 배치와 스트리밍을 동시 실행하기 위해

<details>
<summary>정답 확인</summary>

**정답: B)**

StatementSet은 하나의 파이프라인에서 여러 INSERT 문을 동시에 실행할 수 있게 해줍니다.
동일한 소스 데이터를 서로 다른 조건으로 필터링하여 각각 다른 싱크에 저장할 때 유용합니다.

</details>

### 🏋️ 실습

Table API로 `sample_data`에서 **국가별 최대 tel 값**을 구하고,
SQL로 같은 결과를 만들어 보세요.

```python
# 힌트 1: Table API
# table.group_by(col('country')).select(col('country'), col('tel').cast(...).max)

# 힌트 2: SQL
# t_env.sql_query(f"SELECT ... FROM {table} GROUP BY ...")
```

<details>
<summary>모범 답안</summary>

```python
# Table API
table.group_by(col('country')) \
     .select(col('country'),
             col('tel').cast(DataTypes.BIGINT()).max.alias('max_tel')) \
     .execute().print()

# SQL
t_env.sql_query(f"""
    SELECT JSON_VALUE(`data`, '$.addr.country') as country,
           MAX(CAST(JSON_VALUE(`data`, '$.tel') AS BIGINT)) as max_tel
    FROM {table}
    GROUP BY JSON_VALUE(`data`, '$.addr.country')
""").execute().print()
```

</details>

**Table API 개념 매핑표:**

| Flink Table API | Spark DataFrame | 설명 |
|-----------------|-----------------|------|
| `TableEnvironment` | `SparkSession` | 실행 환경 |
| `Table` | `DataFrame` | 테이블 추상화 |
| `create_temporary_table()` | `createOrReplaceTempView()` | 임시 테이블 등록 |
| `from_path()` | `table()` | 테이블 참조 |
| `execute_sql()` | `sql()` | SQL 실행 |
| `@udtf` | `@udf` + `explode` | 테이블 함수 |
| `group_by()` | `groupBy()` | 그룹화 |
| `execute_insert()` | `write.insertInto()` | 결과 저장 |

---
## 5. DataStream과 Table API 혼합 사용

PyFlink에서는 DataStream API와 Table API를 자유롭게 혼합하여 사용할 수 있습니다.

**혼합 사용 시나리오:**

| 상황 | 추천 API |
|------|---------|
| SQL로 표현 가능한 집계/조인 | Table API |
| ETL 파이프라인 | Table API |
| 복잡한 이벤트 처리 (CEP) | DataStream API |
| 커스텀 상태 관리 필요 | DataStream API |
| 밀리초 단위 지연시간 요구 | DataStream API |
| 빠른 프로토타이핑 | Table API |

**실무 팁:** Table API로 데이터를 정제한 뒤,
DataStream API로 세밀한 로직을 구현하는 패턴이 흔합니다.

In [67]:
# DataStream ↔ Table 변환 예제
env = StreamExecutionEnvironment.get_execution_environment()
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

# 1. Table API로 소스 정의 (datagen: 랜덤 데이터 생성)
t_env.create_temporary_table(
    'source',
    TableDescriptor.for_connector('datagen')
                   .schema(Schema.new_builder()
                           .column('id', DataTypes.BIGINT())
                           .column('data', DataTypes.STRING())
                           .build())
                   .option("number-of-rows", "10")
                   .build()
)

t_env.create_temporary_table(
    'mix_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('result', DataTypes.BIGINT())
                           .build())
                   .build()
)

@udf(result_type=DataTypes.BIGINT())
def length(data):
    return len(data)

# 2. Table API 연산: id + length(data) 계산
table = t_env.from_path("source")
table = table.select(col('id'), length(col('data')))

# 3. Table → DataStream 변환
ds = t_env.to_data_stream(table)

# 4. DataStream API 연산: 두 값 합산
ds = ds.map(lambda i: i[0] + i[1], output_type=Types.LONG())

# 5. DataStream → Table 변환 후 출력
table = t_env.from_data_stream(ds, col("result"))
table.execute_insert('mix_sink').wait()

1> +I[8828122227636467502]
6> +I[1748884498219259495]
7> +I[-3753786784492572021]
3> +I[-1907455819288155062]
10> +I[-601833296558296765]
8> +I[5026942652314736869]
2> +I[-5763115033974670985]
4> +I[-8826528858631339760]


Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    target=lambda: self._read_inputs(elements_iterator),
             

9> +I[-7410871195654950137]
5> +I[406811604106862199]


### 📝 FAQ

**Q. 어떤 상황에서 혼합 사용하나요?**

> 예를 들어, Kafka에서 들어오는 JSON 데이터를 **Table API/SQL**로 파싱하고 필터링한 뒤,
> **DataStream API**로 복잡한 패턴 감지나 커스텀 상태 로직을 구현하는 패턴이 일반적입니다.
> Table API의 편의성과 DataStream API의 유연성을 모두 활용할 수 있습니다.

---

### 🏋️ 실습

Table API로 데이터를 필터링한 후 DataStream으로 변환하여
커스텀 로직을 적용해 보세요.

```python
# 힌트:
# 1. t_env.from_elements()로 테이블 생성
# 2. table.filter()로 조건 필터링
# 3. t_env.to_data_stream(table)로 DataStream 변환
# 4. ds.map()으로 커스텀 변환 적용
# 5. t_env.from_data_stream(ds)로 다시 테이블 변환 후 출력
```

---
## 6. 커넥터 소개

PyFlink는 다양한 외부 시스템과의 연동을 위한 커넥터를 제공합니다.

| 커넥터 | 용도 | 사용 사례 |
|--------|------|----------|
| **Kafka** | 메시지 큐 | 실시간 이벤트 처리, 로그 수집 |
| **Elasticsearch** | 검색 엔진 | 실시간 검색 인덱싱, 모니터링 |
| **JDBC** | 관계형 DB | 참조 데이터 조회, 결과 저장 |
| **FileSystem** | 파일 시스템 | 배치 처리, 로그 아카이빙 |
| **Pulsar** | 메시지 큐 | Kafka 대안, 멀티 테넌시 |

> 실제 커넥터 사용 시에는 해당 커넥터의 JAR 파일이 필요합니다.
> `env.add_jars("file:///path/to/connector.jar")`로 추가합니다.

### Kafka 커넥터 패턴

**DataStream API 방식:**

```python
from pyflink.datastream.connectors.kafka import KafkaSource, KafkaSink

# Source
kafka_source = KafkaSource.builder() \
    .set_bootstrap_servers("localhost:9092") \
    .set_topics("input-topic") \
    .set_group_id("flink-consumer-group") \
    .set_starting_offsets(KafkaOffsetsInitializer.earliest()) \
    .set_value_only_deserializer(SimpleStringSchema()) \
    .build()

ds = env.from_source(kafka_source, WatermarkStrategy.no_watermarks(), "Kafka Source")
```

**Table API / SQL 방식:**

```sql
CREATE TABLE kafka_source (
    event_time TIMESTAMP(3),
    user_id STRING,
    action STRING
) WITH (
    'connector' = 'kafka',
    'topic' = 'user-events',
    'properties.bootstrap.servers' = 'localhost:9092',
    'scan.startup.mode' = 'earliest-offset',
    'format' = 'json'
)
```

> 실제 Kafka 연동 실습은 **stream-lab/** 프로젝트에서 진행합니다.
> stream-lab에서는 Producer → Kafka → Flink → PostgreSQL 전체 파이프라인을 구축합니다.

---
## 7. 최종 정리

### DataStream API vs Table API 최종 비교

| 특성 | DataStream API | Table API |
|------|---------------|----------|
| 추상화 수준 | 저수준 | 고수준 |
| 상태 접근 | 직접 제어 가능 | 자동 관리 |
| 타이머 | 직접 등록 가능 | 불가능 |
| SQL 지원 | 불가능 | 가능 |
| 최적화 | 수동 | 자동 |
| 학습 곡선 | 높음 | 낮음 |
| Spark 대응 | RDD API | DataFrame API |

**언제 어떤 API를 사용할까?**

- **Table API**: SQL/관계형 연산, 빠른 개발, 자동 최적화가 필요한 경우
- **DataStream API**: 세밀한 상태 제어, 복잡한 이벤트 처리, 타이머 사용이 필요한 경우
- **혼합 사용**: Table API로 정제 → DataStream API로 세밀한 로직

### 다음 단계

이 교안에서 PyFlink의 기본 API를 학습했습니다.
다음 단계로 **stream-lab** 실습을 진행합니다.

**stream-lab 실습 내용:**
1. Kafka Producer로 실시간 결제 이벤트 생성
2. Naive Python Consumer vs PyFlink Consumer 비교
3. 윈도우 집계 + PostgreSQL 저장 파이프라인 구축
4. 지연 도착 데이터(Late Data) 처리 전략

<!-- IMAGE: 전체 학습 로드맵 다이어그램 -->

### 참고 자료

- [PyFlink 공식 문서](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/overview/)
- [PyFlink DataStream API 튜토리얼](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/datastream_tutorial/)
- [PyFlink Table API 튜토리얼](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/table_api_tutorial/)
- [Apache Flink GitHub](https://github.com/apache/flink)